# LSEG Data Pull - NetPayout Duration

## 0. Setup

In [1]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import lseg.data as ld

warnings.filterwarnings("ignore", category=FutureWarning, module="lseg")
pd.set_option("display.max_columns", 120)

BASE_DIR = Path("/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data")
DATA_DIR = BASE_DIR / "intermediate"
CACHE_DATA_DIR = BASE_DIR / "cache"
CACHE_DATA_DIR.mkdir(parents=True, exist_ok=True)

import hashlib
import json
import random
import re
import time
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["Times New Roman", "Times", "DejaVu Serif"]


## 1. Input-Parquets laden

In [2]:
EURO500_PATH = DATA_DIR / "euro500.parquet"
EURO500_RETURNS_PATH = DATA_DIR / "euro500_index_returns.parquet"
DAILY_RETURNS_IN_INDEX_PATH = DATA_DIR / "euro500_daily_returns.parquet"

for p in [EURO500_PATH, EURO500_RETURNS_PATH, DAILY_RETURNS_IN_INDEX_PATH]:
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p}")

euro500 = pd.read_parquet(EURO500_PATH)
euro500_returns = pd.read_parquet(EURO500_RETURNS_PATH)
daily_returns_euro500_in_index = pd.read_parquet(DAILY_RETURNS_IN_INDEX_PATH)

print("Loaded:")
print("- euro500:", euro500.shape)
print("- euro500_returns:", euro500_returns.shape)
print("- daily_returns_euro500_in_index:", daily_returns_euro500_in_index.shape)

Loaded:
- euro500: (56500, 25)
- euro500_returns: (7016, 8)
- daily_returns_euro500_in_index: (3457796, 10)


## 2. Modul A: Annual Balance Sheet (FY)

In [3]:
# ------------------------------------------------------------
# Step 5 — Netpayout Table (clean): BE, assets, debt + beta
#   Input  : euro500.parquet
#   Output : euro500_netpayout.parquet
# ------------------------------------------------------------


# -------------------------
# Config
# -------------------------
BASE_PATH = DATA_DIR / 'euro500.parquet'
OUTPUT_PATH = DATA_DIR / 'euro500_netpayout.parquet'
CACHE_DIR = CACHE_DATA_DIR / 'moduleA_cache_by_company_id'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
BAD_LOG_PATH = CACHE_DATA_DIR / 'moduleA_bad_ids.csv'
STEP5_ROWS_PATH = CACHE_DATA_DIR / 'moduleA_step5_rows.parquet'
STEP5_CKPT_PATH = CACHE_DATA_DIR / 'moduleA_step5_checkpoint.json'


TARGET_FIELDS = {
    'BE': 'TR.F.COMEQTOT(Period=FY0)',
    'assets': 'TR.F.TOTASSETS(Period=FY0)',
    'debt': 'TR.F.DEBTTOT(Period=FY0)',
}
TARGET_COLS = list(TARGET_FIELDS.keys())

ASOF_TOL_DAYS = 365
MAX_RETRIES = 2
BASE_SLEEP = 1.0
FORCE_REFRESH = False
CACHE_ONLY = False  # False => allow LSEG pull (cache still used)
CACHE_VERSION = 'v2'
DEBUG_FIRST_RAW = True
DEBUG_SHOWN = {'done': False}
DEBUG_VERBOSE = False
DEBUG_MAX_COMPANIES = 15

BATCH_SIZE = 100
BATCH_PAUSE_SEC = 0
MULTI_UNIVERSE_CHUNK = 25

RATE_LIMIT_EVENTS = 0
LAST_GETDATA_ERR_MSG = ""

# LSEG request governor (to avoid 429 storms)
MIN_CALL_INTERVAL_SEC = 0.7
RATE_LIMIT_COOLDOWN_SEC = 3.0
RATE_LIMIT_MULTIPLIER = 1.5
RATE_LIMIT_HARD_PAUSE_SEC = 10.0
RATE_LIMIT_STREAK_TRIGGER = 2
FAIL_FAST_ON_RATE_LIMIT = True
LAST_CALL_TS = 0.0
GLOBAL_COOLDOWN_UNTIL = 0.0
RATE_LIMIT_STREAK = 0

if not BASE_PATH.exists():
    raise FileNotFoundError(f'Missing file: {BASE_PATH}')

# -------------------------
# Helpers
# -------------------------
def _clean_str(s: pd.Series) -> pd.Series:
    x = s.astype('string').str.strip()
    return x.where(x.notna() & (x != ''), pd.NA)


def _resolve_asof(df: pd.DataFrame) -> pd.Series:
    if 'date' not in df.columns:
        raise ValueError('Missing required date column.')
    return pd.to_datetime(df['date'], errors='coerce').dt.normalize()


def _flatten_cols(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    if isinstance(x.columns, pd.MultiIndex):
        x.columns = [' | '.join([str(v) for v in tup if v is not None]).strip() for tup in x.columns]
    else:
        x.columns = [str(c).strip() for c in x.columns]
    return x


def _build_company_candidates(company_req: pd.DataFrame) -> list[tuple[str, str]]:
    """Collect all unique identifiers for one company across time."""
    q = company_req.copy().sort_values('date')

    out = []
    seen = set()

    def _norm_isin(value):
        if pd.isna(value):
            return ''
        v = str(value).strip()
        if not v:
            return ''
        if v.upper().startswith('ISIN:'):
            v = v.split(':', 1)[1].strip()
        return v

    def _add(id_type: str, value):
        if pd.isna(value):
            return
        v = str(value).strip()
        if not v:
            return
        key = (id_type, v)
        if key in seen:
            return
        seen.add(key)
        out.append(key)

    if 'ISIN' in q.columns:
        for val in q['ISIN']:
            isin = _norm_isin(val)
            if isin:
                _add('ISIN', isin)
                _add('ISIN', f'ISIN:{isin}')

    if 'RIC_current' in q.columns:
        for val in q['RIC_current']:
            _add('RIC', val)

    if 'RIC' in q.columns:
        for val in q['RIC']:
            _add('RIC', val)

    if 'pull_id' in q.columns and 'id_type' in q.columns:
        for id_type, pull_id in zip(q['id_type'], q['pull_id']):
            it = str(id_type).upper().strip() if pd.notna(id_type) else ''
            if it == 'ISIN':
                isin = _norm_isin(pull_id)
                if isin:
                    _add('ISIN', isin)
                    _add('ISIN', f'ISIN:{isin}')
            elif it == 'RIC':
                _add('RIC', pull_id)

    return out


def _cache_file(firm_id: str, id_type: str, pull_id: str) -> Path:
    raw = f'{firm_id}|{id_type}|{pull_id}'
    h = hashlib.sha1(raw.encode('utf-8')).hexdigest()[:12]
    clean = re.sub(r'[^A-Za-z0-9._-]', '_', firm_id)
    preferred = CACHE_DIR / f"{clean[:70]}__{id_type}_{h}__{CACHE_VERSION}.parquet"
    if preferred.exists():
        return preferred

    # Fallback: allow changed firm_id while keeping same (id_type, pull_id) hash.
    matches = sorted(CACHE_DIR.glob(f"*__{id_type}_{h}__{CACHE_VERSION}.parquet"))
    if matches:
        return matches[0]

    # Legacy fallback without explicit cache version suffix.
    legacy = sorted(CACHE_DIR.glob(f"*__{id_type}_{h}.parquet"))
    if legacy:
        return legacy[0]

    return preferred


def _load_cache(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=['date'] + TARGET_COLS)
    d = pd.read_parquet(path).copy()
    d['date'] = pd.to_datetime(d.get('date'), errors='coerce').dt.normalize()
    for c in TARGET_COLS:
        if c not in d.columns:
            d[c] = np.nan
        d[c] = pd.to_numeric(d[c], errors='coerce')
    d = d.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return d[['date'] + TARGET_COLS]


def _save_cache(path: Path, d: pd.DataFrame) -> None:
    out = d.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce').dt.normalize()
    for c in TARGET_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    tmp = path.with_suffix(path.suffix + '.tmp')
    out.to_parquet(tmp, index=False)
    tmp.replace(path)


def _extract_history(raw: pd.DataFrame, value_col_name: str) -> pd.DataFrame:
    if raw is None or len(raw) == 0:
        return pd.DataFrame(columns=['date', value_col_name])

    x = _flatten_cols(pd.DataFrame(raw).reset_index())
    if x.empty:
        return pd.DataFrame(columns=['date', value_col_name])

    date_col = None
    for c in x.columns:
        if 'date' in c.lower() or c.lower() == 'date':
            date_col = c
            break
    if date_col is None:
        date_col = x.columns[0]

    id_like = set()
    for c in x.columns:
        cl = c.lower()
        if c == date_col:
            continue
        if cl in {'instrument', 'ric', 'isin'} or 'instrument' in cl or cl.endswith('ric') or cl.endswith('isin'):
            id_like.add(c)

    value_candidates = [c for c in x.columns if c != date_col and c not in id_like]
    if not value_candidates:
        return pd.DataFrame(columns=['date', value_col_name])

    best = None
    best_n = -1
    for c in value_candidates:
        n = int(pd.to_numeric(x[c], errors='coerce').notna().sum())
        if n > best_n:
            best_n = n
            best = c

    if best is None or best_n <= 0:
        return pd.DataFrame(columns=['date', value_col_name])

    out = pd.DataFrame({
        'date': pd.to_datetime(x[date_col], errors='coerce').dt.normalize(),
        value_col_name: pd.to_numeric(x[best], errors='coerce'),
    })
    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return out[['date', value_col_name]]


def _govern_requests() -> None:
    global LAST_CALL_TS, GLOBAL_COOLDOWN_UNTIL
    now = time.time()
    if now < GLOBAL_COOLDOWN_UNTIL:
        wait = GLOBAL_COOLDOWN_UNTIL - now
        if wait > 0:
            time.sleep(wait)
    now = time.time()
    gap = now - LAST_CALL_TS
    if gap < MIN_CALL_INTERVAL_SEC:
        time.sleep(MIN_CALL_INTERVAL_SEC - gap)


def _cache_covers_range(cached: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> bool:
    if cached is None or cached.empty:
        return False
    cmin = cached['date'].min()
    cmax = cached['date'].max()
    has_any_cached_value = bool(cached[TARGET_COLS].notna().any().any()) if not cached.empty else False
    return bool(pd.notna(cmin) and pd.notna(cmax) and cmin <= start and cmax >= end and has_any_cached_value)


def _call_get_data(universe: list[str], fields: list[str], params: dict, dbg_label: str) -> pd.DataFrame:
    global LAST_CALL_TS, GLOBAL_COOLDOWN_UNTIL, RATE_LIMIT_STREAK, RATE_LIMIT_EVENTS, LAST_GETDATA_ERR_MSG

    LAST_GETDATA_ERR_MSG = ""
    last_err = None
    raw = None
    for r in range(MAX_RETRIES):
        try:
            _govern_requests()
            raw = ld.get_data(universe=universe, fields=fields, parameters=params)
            LAST_CALL_TS = time.time()
            RATE_LIMIT_STREAK = 0
            LAST_GETDATA_ERR_MSG = ""
            break
        except Exception as e:
            LAST_CALL_TS = time.time()
            last_err = e
            LAST_GETDATA_ERR_MSG = str(e)
            msg = str(e)
            is_rate_limit = 'Too many requests' in msg or '429' in msg
            if is_rate_limit:
                RATE_LIMIT_STREAK += 1
                RATE_LIMIT_EVENTS += 1
                cooldown = RATE_LIMIT_COOLDOWN_SEC * (RATE_LIMIT_MULTIPLIER ** r) + random.random()
                if RATE_LIMIT_STREAK >= RATE_LIMIT_STREAK_TRIGGER:
                    cooldown = max(cooldown, RATE_LIMIT_HARD_PAUSE_SEC)
                GLOBAL_COOLDOWN_UNTIL = max(GLOBAL_COOLDOWN_UNTIL, time.time() + cooldown)
                if DEBUG_VERBOSE:
                    print(f'[DEBUG rate limit] id={dbg_label} retry={r+1}/{MAX_RETRIES} streak={RATE_LIMIT_STREAK} cooldown={cooldown:.1f}s err={type(e).__name__}: {e}')
                if FAIL_FAST_ON_RATE_LIMIT:
                    break
                time.sleep(cooldown)
            else:
                RATE_LIMIT_STREAK = 0
                if DEBUG_VERBOSE:
                    print(f'[DEBUG get_data error] id={dbg_label} retry={r+1}/{MAX_RETRIES} err={type(e).__name__}: {e}')
                time.sleep(BASE_SLEEP * (2 ** r) + random.random() * 0.3)

    if raw is None or len(raw) == 0:
        if DEBUG_VERBOSE:
            if last_err is not None:
                print(f'[DEBUG empty raw after errors] id={dbg_label} last_err={type(last_err).__name__}: {last_err} (skipped)')
            else:
                print(f'[DEBUG empty raw no exception] id={dbg_label}')
        return pd.DataFrame()

    return pd.DataFrame(raw)


def _hist_from_flat_frame(x: pd.DataFrame) -> pd.DataFrame:
    if x is None or x.empty:
        return pd.DataFrame(columns=['date'] + TARGET_COLS)

    x = _flatten_cols(x)
    if x.empty:
        return pd.DataFrame(columns=['date'] + TARGET_COLS)

    date_col = None
    for c in x.columns:
        uc = c.upper()
        if 'PERIOD' in uc and 'DATE' in uc:
            date_col = c
            break
    if date_col is None:
        for c in x.columns:
            if 'date' in c.lower():
                date_col = c
                break
    if date_col is None:
        return pd.DataFrame(columns=['date'] + TARGET_COLS)

    out = pd.DataFrame({'date': pd.to_datetime(x[date_col], errors='coerce').dt.normalize()})

    def _norm(s: str) -> str:
        return re.sub(r'[^A-Z0-9]', '', str(s).upper())

    def _field_token(field_expr: str) -> str:
        m = re.search(r'TR\.F\.([A-Z0-9_]+)', str(field_expr), flags=re.IGNORECASE)
        return _norm(m.group(1)) if m else _norm(field_expr)

    fallback_tokens = {
        'BE': ['COMEQTOT', 'COMEQ', 'EQUITY', 'COMMONEQUITY', 'BOOKEQUITY'],
        'assets': ['TOTASSETS', 'TOTALASSETS', 'ASSETS'],
        'debt': ['DEBTTOT', 'TOTALDEBT', 'DEBT'],
    }

    norm_cols = {c: _norm(c) for c in x.columns}

    for tgt, field_expr in TARGET_FIELDS.items():
        token = _field_token(field_expr)
        cand = None
        for c, cn in norm_cols.items():
            if token and (token in cn or cn in token):
                cand = c
                break
        if cand is None:
            for alt in fallback_tokens.get(tgt, []):
                alt_n = _norm(alt)
                for c, cn in norm_cols.items():
                    if alt_n and alt_n in cn:
                        cand = c
                        break
                if cand is not None:
                    break
        out[tgt] = pd.to_numeric(x[cand], errors='coerce') if cand is not None else np.nan

    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return out[['date'] + TARGET_COLS]


def _pull_targets_segment(pull_id: str, start: pd.Timestamp, end: pd.Timestamp) -> pd.DataFrame:
    if pd.isna(start) or pd.isna(end) or start > end:
        return pd.DataFrame(columns=['date'] + TARGET_COLS)

    params = {
        'SDate': start.strftime('%Y-%m-%d'),
        'EDate': end.strftime('%Y-%m-%d'),
        'Frq': 'FY',
    }
    fields = ['TR.F.PeriodEndDate(Period=FY0)'] + list(TARGET_FIELDS.values())

    raw_df = _call_get_data(universe=[pull_id], fields=fields, params=params, dbg_label=pull_id)
    if raw_df.empty:
        return pd.DataFrame(columns=['date'] + TARGET_COLS)

    if DEBUG_FIRST_RAW and (not DEBUG_SHOWN['done']):
        print('DEBUG first raw get_data shape:', raw_df.shape)
        print('DEBUG first raw columns:', list(raw_df.columns)[:20])
        try:
            nn = raw_df.notna().sum()
            print('DEBUG first raw non-null counts (top):')
            print(nn.head(20))
        except Exception:
            pass
        DEBUG_SHOWN['done'] = True

    return _hist_from_flat_frame(raw_df)


def _pull_targets_multi_segment(pull_ids: list[str], start: pd.Timestamp, end: pd.Timestamp) -> dict[str, pd.DataFrame]:
    if not pull_ids or pd.isna(start) or pd.isna(end) or start > end:
        return {}

    uniq = []
    seen = set()
    for pid in pull_ids:
        v = str(pid).strip()
        if not v:
            continue
        k = v.upper()
        if k in seen:
            continue
        seen.add(k)
        uniq.append(v)

    if not uniq:
        return {}

    params = {
        'SDate': start.strftime('%Y-%m-%d'),
        'EDate': end.strftime('%Y-%m-%d'),
        'Frq': 'FY',
    }
    fields = ['TR.F.PeriodEndDate(Period=FY0)'] + list(TARGET_FIELDS.values())

    out = {pid: pd.DataFrame(columns=['date'] + TARGET_COLS) for pid in uniq}

    def _assign_from_raw(ids: list[str], raw_df: pd.DataFrame) -> None:
        if raw_df is None or raw_df.empty:
            return
        x = _flatten_cols(raw_df)
        if x.empty:
            return

        inst_col = None
        for c in x.columns:
            cl = c.lower()
            if 'instrument' in cl or cl in {'ric', 'isin'}:
                inst_col = c
                break

        if inst_col is None:
            if len(ids) == 1:
                out[ids[0]] = _hist_from_flat_frame(x)
            return

        inst_norm = x[inst_col].astype('string').str.strip().str.upper()
        for pid in ids:
            g = x.loc[inst_norm == str(pid).upper()].copy()
            if g.empty:
                continue
            out[pid] = _hist_from_flat_frame(g)

    def _periodend_collect_error(msg: str) -> bool:
        m = (msg or '').lower()
        return ('unable to collect data for the field' in m) and ('periodenddate' in m)

    def _fetch_ids(ids: list[str]) -> None:
        if not ids:
            return
        label = f'multi[{len(ids)}]:' + ','.join(ids[:3])
        raw_df = _call_get_data(universe=ids, fields=fields, params=params, dbg_label=label)

        if raw_df.empty:
            err = LAST_GETDATA_ERR_MSG
            # Some bad identifiers can poison a whole multi request: split recursively.
            if len(ids) > 1 and _periodend_collect_error(err):
                mid = len(ids) // 2
                left = ids[:mid]
                right = ids[mid:]
                if DEBUG_VERBOSE:
                    print(f'[DEBUG multi split] size={len(ids)} -> {len(left)}+{len(right)} due to PeriodEndDate collect error')
                _fetch_ids(left)
                _fetch_ids(right)
            return

        _assign_from_raw(ids, raw_df)

    for i in range(0, len(uniq), MULTI_UNIVERSE_CHUNK):
        chunk = uniq[i:i+MULTI_UNIVERSE_CHUNK]
        _fetch_ids(chunk)

    return out


def _combine(frames: list[pd.DataFrame]) -> pd.DataFrame:

    recs = []
    for f in frames:
        if f is None or not isinstance(f, pd.DataFrame) or f.empty:
            continue
        g = f.copy()
        for c in ['date'] + TARGET_COLS:
            if c not in g.columns:
                g[c] = np.nan if c != 'date' else pd.NaT
        g = g[['date'] + TARGET_COLS]
        g = g.loc[~g.isna().all(axis=1)]
        if g.empty:
            continue
        recs.extend(g.to_dict('records'))

    if not recs:
        return pd.DataFrame(columns=['date'] + TARGET_COLS)

    out = pd.DataFrame.from_records(recs, columns=['date'] + TARGET_COLS)
    out['date'] = pd.to_datetime(out['date'], errors='coerce').dt.normalize()
    for c in TARGET_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    return out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')


def _update_cache(firm_id: str, id_type: str, pull_id: str, start: pd.Timestamp, end: pd.Timestamp, force_refresh: bool = False) -> pd.DataFrame:
    path = _cache_file(firm_id, id_type, pull_id)
    cached = pd.DataFrame(columns=['date'] + TARGET_COLS) if force_refresh else _load_cache(path)

    segments = []
    if cached.empty:
        segments.append((start, end))
    else:
        cmin = cached['date'].min()
        cmax = cached['date'].max()
        has_any_cached_value = bool(cached[TARGET_COLS].notna().any().any()) if not cached.empty else False
        # short-circuit only if cache covers range AND contains at least one real value
        if pd.notna(cmin) and pd.notna(cmax) and cmin <= start and cmax >= end and has_any_cached_value and (not force_refresh):
            return cached
        if start < cmin:
            segments.append((start, cmin - pd.Timedelta(days=1)))
        if end > cmax:
            segments.append((cmax + pd.Timedelta(days=1), end))

    pulled = []
    for sdt, edt in segments:
        if sdt > edt:
            continue
        seg = _pull_targets_segment(pull_id, sdt, edt)
        if not seg.empty:
            pulled.append(seg)

    combined = _combine([cached] + pulled)
    if not combined.empty or force_refresh:
        _save_cache(path, combined)

    out = _load_cache(path)
    return out if not out.empty else combined


def _map_to_asof(req_dates: pd.Series, hist: pd.DataFrame, tol_days: int) -> pd.DataFrame:
    left = pd.DataFrame({'date': pd.to_datetime(req_dates, errors='coerce').dt.normalize()}).dropna().sort_values('date')
    if left.empty:
        return pd.DataFrame(columns=['date'] + TARGET_COLS)

    if hist is None or hist.empty:
        for c in TARGET_COLS:
            left[c] = np.nan
        return left

    right = hist[['date'] + TARGET_COLS].copy().sort_values('date').drop_duplicates(['date'], keep='last')
    out = pd.merge_asof(
        left,
        right,
        on='date',
        direction='backward',
        tolerance=pd.Timedelta(days=ASOF_TOL_DAYS),
    )
    return out


rebuild_output = FORCE_REFRESH or (locals().get('N', 0) > 0) or (not STEP5_ROWS_PATH.exists()) or (not OUTPUT_PATH.exists())
if (not rebuild_output) and BASE_PATH.exists() and OUTPUT_PATH.exists():
    rebuild_output = BASE_PATH.stat().st_mtime > OUTPUT_PATH.stat().st_mtime


def _build_step5_rows_from_cache(base_df: pd.DataFrame) -> pd.DataFrame:
    req_cols = [c for c in ['firm_id', 'date', 'ISIN', 'RIC_current', 'RIC', 'id_type', 'pull_id'] if c in base_df.columns]
    req = (
        base_df[req_cols]
        .dropna(subset=['firm_id', 'date'])
        .drop_duplicates(['firm_id', 'date'], keep='last')
        .copy()
    )
    if req.empty:
        return pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS)

    rows = []
    groups = req.groupby('firm_id', sort=False)
    total = len(groups)
    for ix, (firm_id, g) in enumerate(groups, start=1):
        dates = pd.to_datetime(g['date'], errors='coerce').dropna().sort_values().unique()
        panel = pd.DataFrame({'date': pd.to_datetime(dates)})
        for c in TARGET_COLS:
            panel[c] = np.nan

        # Dynamic candidate list per company using the same normalization as in main pull logic.
        uniq = _build_company_candidates(g)

        for id_type, pull_id in uniq:
            hist = _load_cache(_cache_file(str(firm_id), str(id_type), str(pull_id)))
            if hist.empty:
                continue
            mapped = _map_to_asof(panel['date'], hist, ASOF_TOL_DAYS)
            if mapped.empty:
                continue
            for c in TARGET_COLS:
                if c in mapped.columns:
                    panel[c] = pd.to_numeric(panel[c], errors='coerce').combine_first(pd.to_numeric(mapped[c], errors='coerce'))
            if panel[TARGET_COLS].notna().all(axis=1).all():
                break

        panel.insert(0, 'firm_id', str(firm_id))
        rows.append(panel[['firm_id', 'date'] + TARGET_COLS])

        if ix % 100 == 0 or ix == total:
            found = int(panel[TARGET_COLS].notna().all(axis=1).sum())
            print(f'[Step5 cache rebuild] {ix}/{total} company={str(firm_id)[:40]} resolved_rows={found}/{len(panel)}')

    if not rows:
        return pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS)

    out = pd.concat(rows, ignore_index=True)
    out['date'] = pd.to_datetime(out['date'], errors='coerce').dt.normalize()
    for c in TARGET_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce')
    out = out.sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last')
    return out


# -------------------------
# Pull loop (batched + resume)
# -------------------------
if 'base' not in locals():
    base = pd.read_parquet(BASE_PATH).copy()
    if 'date' not in base.columns:
        base['date'] = _resolve_asof(base)
    else:
        base['date'] = pd.to_datetime(base['date'], errors='coerce').dt.normalize()

    for c in ['firm_id', 'ISIN', 'RIC_current', 'RIC', 'pull_id', 'id_type']:
        if c in base.columns:
            base[c] = _clean_str(base[c])

    if 'id_type' not in base.columns:
        base['id_type'] = np.select(
            [base.get('ISIN', pd.Series(pd.NA, index=base.index)).notna(),
             base.get('RIC_current', pd.Series(pd.NA, index=base.index)).notna(),
             base.get('RIC', pd.Series(pd.NA, index=base.index)).notna()],
            ['ISIN', 'RIC', 'RIC'],
            default=pd.NA,
        )
    if 'pull_id' not in base.columns:
        base['pull_id'] = np.select(
            [base.get('ISIN', pd.Series(pd.NA, index=base.index)).notna(),
             base.get('RIC_current', pd.Series(pd.NA, index=base.index)).notna(),
             base.get('RIC', pd.Series(pd.NA, index=base.index)).notna()],
            [base.get('ISIN', pd.Series(pd.NA, index=base.index)),
             base.get('RIC_current', pd.Series(pd.NA, index=base.index)),
             base.get('RIC', pd.Series(pd.NA, index=base.index))],
            default=pd.NA,
        )

    base['id_type'] = _clean_str(base['id_type'])
    base['pull_id'] = _clean_str(base['pull_id'])

    if 'firm_id' not in base.columns:
        base['firm_id'] = pd.Series(pd.NA, index=base.index, dtype='string')

req_cols = [c for c in ['firm_id', 'date', 'ISIN', 'RIC_current', 'RIC', 'id_type', 'pull_id'] if c in base.columns]
req = (
    base[req_cols]
    .dropna(subset=['firm_id', 'date'])
    .drop_duplicates(['firm_id', 'date'], keep='last')
    .reset_index(drop=True)
)
if req.empty:
    raise ValueError('No valid (firm_id, date) rows found for Step 5.')

company_candidates_map = {}
for ck, g in req.groupby('firm_id', sort=False):
    company_candidates_map[str(ck)] = _build_company_candidates(g)

companies_all = req['firm_id'].dropna().astype(str).unique().tolist()
N_total = len(companies_all)

existing_step_rows = pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS)
if STEP5_ROWS_PATH.exists():
    try:
        existing_step_rows = pd.read_parquet(STEP5_ROWS_PATH)
        if 'date' in existing_step_rows.columns:
            existing_step_rows['date'] = pd.to_datetime(existing_step_rows['date'], errors='coerce').dt.normalize()
        for c in TARGET_COLS:
            if c not in existing_step_rows.columns:
                existing_step_rows[c] = np.nan
            existing_step_rows[c] = pd.to_numeric(existing_step_rows[c], errors='coerce')
        existing_step_rows = existing_step_rows[['firm_id', 'date'] + TARGET_COLS]
    except Exception as e:
        print(f'Warning: failed to read STEP5 rows cache, continuing empty: {e}')
        existing_step_rows = pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS)

processed_from_rows = set(existing_step_rows['firm_id'].dropna().astype(str).unique().tolist()) if not existing_step_rows.empty else set()
processed_from_ckpt = set()
if STEP5_CKPT_PATH.exists():
    try:
        ck = json.loads(STEP5_CKPT_PATH.read_text())
        processed_from_ckpt = set(str(x) for x in ck.get('processed_companies', []) if str(x).strip())
    except Exception as e:
        print(f'Warning: failed to read checkpoint, ignoring: {e}')

cache_file_count = len(list(CACHE_DIR.glob('*.parquet')))
if cache_file_count == 0 and (processed_from_rows or processed_from_ckpt):
    print('Step5: cache directory is empty -> ignoring step rows/checkpoint and starting full rebuild.')
    processed_from_rows = set()
    processed_from_ckpt = set()

processed_companies = set(processed_from_rows) | set(processed_from_ckpt)
companies = [c for c in companies_all if str(c) not in processed_companies]
N = len(companies)

print('Resume info: total_companies=', N_total, 'already_processed=', len(processed_companies), 'remaining=', N, 'cache_files=', cache_file_count)
print('Step 5 request rows (company x asof):', len(req))
print('As-of range:', req['date'].min(), 'to', req['date'].max())
print('Unique companies:', req['firm_id'].nunique())
print('Active LSEG fields:', TARGET_FIELDS)
print('Mode:', 'CACHE_ONLY' if CACHE_ONLY else 'CACHE+NETWORK')
print('ID fallback order (historical per firm): all ISINs -> all RIC_current -> all RIC (+ pull_id/id_type history)')

run_t0 = time.time()
new_rows_out = []
bad_rows = []

total_cand_calls = 0
total_all3_resolved = 0
total_not_all3 = 0
total_item_found = {c: 0 for c in TARGET_COLS}

if not CACHE_ONLY:
    ld.open_session()
try:
    if N == 0:
        print('No remaining companies to pull in Step 5.')

    n_batches = int(np.ceil(N / BATCH_SIZE)) if N > 0 else 0
    for b_ix, b_start in enumerate(range(0, N, BATCH_SIZE), start=1):
        b_end = min(N, b_start + BATCH_SIZE)
        batch_companies = companies[b_start:b_end]
        batch_t0 = time.time()
        batch_new_rows = []
        batch_processed = []

        print(f'[BATCH {b_ix}/{n_batches}] companies={len(batch_companies)} idx={b_start+1}-{b_end}')

        for k, firm_id in enumerate(batch_companies, start=1):
            q = req[req['firm_id'] == firm_id].copy().sort_values('date')
            dates = q['date'].dropna().drop_duplicates().sort_values().reset_index(drop=True)
            if dates.empty:
                continue

            start = pd.to_datetime(dates.min()).normalize()
            end = pd.to_datetime(dates.max()).normalize()

            panel = pd.DataFrame({'date': dates})
            for c in TARGET_COLS:
                panel[c] = np.nan

            cands = company_candidates_map.get(str(firm_id), [])
            cand_used = 0
            attempted_ids = []

            for cand in cands:
                if panel[TARGET_COLS].notna().all(axis=1).all():
                    break
                if (not isinstance(cand, (list, tuple))) or len(cand) != 2:
                    continue

                id_type = str(cand[0]).upper().strip()
                pull_id = str(cand[1]).strip()
                if not id_type or not pull_id:
                    continue

                cand_used += 1
                total_cand_calls += 1
                attempted_ids.append(pull_id if str(pull_id).upper().startswith('ISIN:') else f'{id_type}:{pull_id}')

                cache_path = _cache_file(str(firm_id), id_type, pull_id)
                cached_pre = _load_cache(cache_path)
                if _cache_covers_range(cached_pre, start, end) and (not FORCE_REFRESH):
                    hist = cached_pre
                else:
                    hist = _update_cache(
                        firm_id=str(firm_id),
                        id_type=id_type,
                        pull_id=pull_id,
                        start=start,
                        end=end,
                        force_refresh=FORCE_REFRESH,
                    )

                if hist is None or hist.empty:
                    continue

                mapped = _map_to_asof(panel['date'], hist, ASOF_TOL_DAYS)
                if mapped is None or mapped.empty:
                    continue

                for c in TARGET_COLS:
                    if c in mapped.columns:
                        panel[c] = pd.to_numeric(panel[c], errors='coerce').combine_first(pd.to_numeric(mapped[c], errors='coerce'))

            panel['firm_id'] = str(firm_id)
            panel = panel[['firm_id', 'date'] + TARGET_COLS]

            row_all = panel[TARGET_COLS].notna().all(axis=1)
            all3_resolved = int(row_all.sum())
            not_all3 = int((~row_all).sum())
            total_all3_resolved += all3_resolved
            total_not_all3 += not_all3

            item_found = {c: int(panel[c].notna().sum()) for c in TARGET_COLS}
            for c in TARGET_COLS:
                total_item_found[c] += item_found[c]

            batch_new_rows.extend(panel.to_dict('records'))
            batch_processed.append(str(firm_id))

            unresolved = panel[panel[TARGET_COLS].isna().any(axis=1)]
            if not unresolved.empty:
                bad_rows.extend(
                    unresolved[['firm_id', 'date']]
                    .assign(reason='not_all3_resolved', n_candidates=int(len(cands)))
                    .to_dict('records')
                )

            elapsed = time.time() - run_t0
            print(
                f'[BATCH {b_ix}/{n_batches}] [{b_start+k}/{N}] company={str(firm_id)[:40]} rows={len(dates)} '
                f'cand_used={cand_used}/{len(cands)} all3_resolved={all3_resolved} not_all3={not_all3} '
                f'found_BE={item_found.get("BE",0)} found_assets={item_found.get("assets",0)} found_debt={item_found.get("debt",0)} '
                f'elapsed={elapsed/60:.1f}m'
            )

        # Persist batch rows for resume
        if batch_new_rows:
            batch_df = pd.DataFrame(batch_new_rows)
            batch_df['date'] = pd.to_datetime(batch_df['date'], errors='coerce').dt.normalize()
            for c in TARGET_COLS:
                batch_df[c] = pd.to_numeric(batch_df[c], errors='coerce')
            batch_df = batch_df.dropna(subset=['firm_id', 'date'])

            if STEP5_ROWS_PATH.exists():
                prev = pd.read_parquet(STEP5_ROWS_PATH)
                if prev is None or prev.empty:
                    combined = batch_df.copy()
                else:
                    prev = prev.copy()
                    prev['date'] = pd.to_datetime(prev.get('date'), errors='coerce').dt.normalize()
                    for c in TARGET_COLS:
                        if c not in prev.columns:
                            prev[c] = np.nan
                        prev[c] = pd.to_numeric(prev[c], errors='coerce')
                    prev = prev[['firm_id', 'date'] + TARGET_COLS].dropna(subset=['firm_id', 'date'])
                    combined = pd.concat([prev, batch_df], ignore_index=True)
            else:
                combined = batch_df.copy()

            combined = combined.sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last')
            combined.to_parquet(STEP5_ROWS_PATH, index=False)
            new_rows_out = combined.to_dict('records')

        if batch_processed:
            processed_companies.update(batch_processed)
            ckpt_payload = {
                'updated_at': pd.Timestamp.utcnow().isoformat(),
                'processed_companies': sorted(processed_companies),
                'rows': int(len(new_rows_out)),
                'remaining_companies': max(0, N_total - len(processed_companies)),
            }
            STEP5_CKPT_PATH.write_text(json.dumps(ckpt_payload, ensure_ascii=False, indent=2))

        if BATCH_PAUSE_SEC > 0 and b_ix < n_batches:
            time.sleep(BATCH_PAUSE_SEC)

finally:
    try:
        if not CACHE_ONLY:
            ld.close_session()
    except Exception:
        pass

print(
    f'Done Step 5: total_companies={N_total}, run_remaining_start={N}, candidate_calls={total_cand_calls}, '
    f'all3_resolved_rows={total_all3_resolved}, not_all3_rows={total_not_all3}, '
    + ', '.join([f'found_{c}={total_item_found[c]}' for c in TARGET_COLS])
)


# Build final output table
if rebuild_output and STEP5_ROWS_PATH.exists():
    out_panel = pd.read_parquet(STEP5_ROWS_PATH)
else:
    out_panel = pd.DataFrame(locals().get('new_rows_out', []))

# If existing step rows are present but contain no usable values, force cache rebuild.
if rebuild_output and (not out_panel.empty):
    _nonnull_any = False
    for _c in TARGET_COLS:
        if _c in out_panel.columns and pd.to_numeric(out_panel[_c], errors='coerce').notna().any():
            _nonnull_any = True
            break
    if not _nonnull_any:
        out_panel = pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS)

if rebuild_output and out_panel.empty:
    # Rebuild from moduleA caches (no LSEG calls).
    _base_tmp = pd.read_parquet(BASE_PATH).copy()
    if 'date' not in _base_tmp.columns:
        _base_tmp['date'] = _resolve_asof(_base_tmp)
    else:
        _base_tmp['date'] = pd.to_datetime(_base_tmp['date'], errors='coerce').dt.normalize()
    for _idc in ['firm_id', 'ISIN', 'RIC_current', 'RIC']:
        if _idc in _base_tmp.columns:
            _base_tmp[_idc] = _clean_str(_base_tmp[_idc])

    if 'firm_id' not in _base_tmp.columns:
        _base_tmp['firm_id'] = _clean_str(_base_tmp.get('firm_id', pd.Series(pd.NA, index=_base_tmp.index)))

    out_panel = _build_step5_rows_from_cache(_base_tmp)
elif rebuild_output:
    out_panel['date'] = pd.to_datetime(out_panel['date'], errors='coerce').dt.normalize()
    for c in TARGET_COLS:
        if c not in out_panel.columns:
            out_panel[c] = np.nan
        out_panel[c] = pd.to_numeric(out_panel[c], errors='coerce')
    out_panel = out_panel[['firm_id', 'date'] + TARGET_COLS]
    out_panel = out_panel.sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last')

# Persist Step-5 resume artifacts (rows + checkpoint) so resume works after restarts.
if rebuild_output:
    out_panel.to_parquet(STEP5_ROWS_PATH, index=False)
    ckpt = {
        'updated_at': pd.Timestamp.utcnow().isoformat(),
        'rows': int(len(out_panel)),
        'processed_companies': sorted(set(out_panel['firm_id'].dropna().astype(str).unique().tolist())) if 'firm_id' in out_panel.columns else [],
    }
    with open(STEP5_CKPT_PATH, 'w', encoding='utf-8') as f:
        json.dump(ckpt, f)
    print('Step5 rows cache:', STEP5_ROWS_PATH, 'rows:', len(out_panel))
    print('Step5 checkpoint:', STEP5_CKPT_PATH)

if 'base' not in locals():
    base = pd.read_parquet(BASE_PATH).copy()
    if 'date' not in base.columns:
        base['date'] = _resolve_asof(base)
    else:
        base['date'] = pd.to_datetime(base['date'], errors='coerce').dt.normalize()

    for c in ['firm_id', 'ISIN', 'RIC_current', 'RIC', 'pull_id', 'id_type', 'firm_id']:
        if c in base.columns:
            base[c] = _clean_str(base[c])

    if 'id_type' not in base.columns:
        base['id_type'] = np.select(
            [base.get('ISIN', pd.Series(pd.NA, index=base.index)).notna(),
             base.get('RIC_current', pd.Series(pd.NA, index=base.index)).notna(),
             base.get('RIC', pd.Series(pd.NA, index=base.index)).notna()],
            ['ISIN', 'RIC', 'RIC'],
            default=pd.NA,
        )
    if 'pull_id' not in base.columns:
        base['pull_id'] = np.select(
            [base.get('ISIN', pd.Series(pd.NA, index=base.index)).notna(),
             base.get('RIC_current', pd.Series(pd.NA, index=base.index)).notna(),
             base.get('RIC', pd.Series(pd.NA, index=base.index)).notna()],
            [base.get('ISIN', pd.Series(pd.NA, index=base.index)),
             base.get('RIC_current', pd.Series(pd.NA, index=base.index)),
             base.get('RIC', pd.Series(pd.NA, index=base.index))],
            default=pd.NA,
        )

    base['id_type'] = _clean_str(base['id_type']) if 'id_type' in base.columns else pd.Series(pd.NA, index=base.index, dtype='string')
    base['pull_id'] = _clean_str(base['pull_id']) if 'pull_id' in base.columns else pd.Series(pd.NA, index=base.index, dtype='string')

    if 'firm_id' not in base.columns:
        base['firm_id'] = base['id_type'].astype('string') + ':' + base['pull_id'].astype('string')
        base.loc[base['pull_id'].isna(), 'firm_id'] = pd.NA
    else:
        base['firm_id'] = _clean_str(base['firm_id'])

if rebuild_output:
    # Stage 1: preferred merge on firm_id + date
    if 'firm_id' in base.columns:
        out_panel_firm = out_panel.rename(columns={'firm_id': 'firm_id'})
        out = base.merge(out_panel_firm, on=['firm_id', 'date'], how='left', suffixes=('', '_firm'))
    else:
        out = base.copy()

    # Stage 2 fallback: merge on firm_id + date
    add = base.merge(out_panel, on=['firm_id', 'date'], how='left', suffixes=('', '_key'))

    for c in TARGET_COLS:
        old_s = pd.to_numeric(out[c], errors='coerce') if c in out.columns else pd.Series(np.nan, index=out.index)
        new_firm_s = pd.to_numeric(out[f'{c}_firm'] if f'{c}_firm' in out.columns else pd.Series(np.nan, index=out.index), errors='coerce')
        new_key_s = pd.to_numeric(add[f'{c}_key'] if f'{c}_key' in add.columns else pd.Series(np.nan, index=out.index), errors='coerce')
        out[c] = new_firm_s.combine_first(new_key_s).combine_first(old_s)
        if f'{c}_firm' in out.columns:
            out = out.drop(columns=[f'{c}_firm'])

if rebuild_output:
    out.to_parquet(OUTPUT_PATH, index=False)

    print('Saved netpayout output:', OUTPUT_PATH, 'rows:', len(out))
    euro500_netpayout_df = out.copy()
    print('Data Wrangler variable ready: euro500_netpayout_df')
else:
    out = pd.read_parquet(OUTPUT_PATH)
    print('Skipped Step-5 rebuild (already up-to-date):', OUTPUT_PATH, 'rows:', len(out))
    euro500_netpayout_df = out.copy()
    print('Data Wrangler variable ready: euro500_netpayout_df')

if locals().get('bad_rows', []):
    bad_df = pd.DataFrame(locals().get('bad_rows', []))
    if BAD_LOG_PATH.exists():
        old = pd.read_csv(BAD_LOG_PATH)
        for c in ['firm_id', 'date', 'reason', 'n_candidates']:
            if c not in old.columns:
                old[c] = pd.NA
            if c not in bad_df.columns:
                bad_df[c] = pd.NA
        out_bad = pd.DataFrame.from_records(
            old[['firm_id', 'date', 'reason', 'n_candidates']].to_dict('records')
            + bad_df[['firm_id', 'date', 'reason', 'n_candidates']].to_dict('records'),
            columns=['firm_id', 'date', 'reason', 'n_candidates'],
        )
    else:
        out_bad = bad_df

    out_bad['date'] = pd.to_datetime(out_bad['date'], errors='coerce').dt.normalize()
    out_bad = out_bad.drop_duplicates(subset=['firm_id', 'date', 'reason'], keep='last')
    out_bad.to_csv(BAD_LOG_PATH, index=False)
    print('Updated bad-id log:', BAD_LOG_PATH, 'rows:', len(out_bad))














Resume info: total_companies= 1248 already_processed= 1166 remaining= 84 cache_files= 1666
Step 5 request rows (company x asof): 56500
As-of range: 1997-12-31 00:00:00 to 2025-12-31 00:00:00
Unique companies: 1248
Active LSEG fields: {'BE': 'TR.F.COMEQTOT(Period=FY0)', 'assets': 'TR.F.TOTASSETS(Period=FY0)', 'debt': 'TR.F.DEBTTOT(Period=FY0)'}
Mode: CACHE+NETWORK
ID fallback order (historical per firm): all ISINs -> all RIC_current -> all RIC (+ pull_id/id_type history)
[BATCH 1/1] companies=84 idx=1-84
DEBUG first raw get_data shape: (1, 5)
DEBUG first raw columns: ['Instrument', 'Period End Date', 'Common Equity - Total', 'Total Assets', 'Debt - Total']
DEBUG first raw non-null counts (top):
Instrument               1
Period End Date          1
Common Equity - Total    1
Total Assets             1
Debt - Total             1
dtype: int64
[BATCH 1/1] [1/84] company=FIRM0000737 rows=1 cand_used=4/4 all3_resolved=0 not_all3=1 found_BE=0 found_assets=0 found_debt=0 elapsed=0.2m
[BATCH 1/1

KeyboardInterrupt: 

### 2B. Coverage-Analyse Modul A (`BE`, `assets`, `debt`)

In [ ]:

# Robust path resolution for Step 5B (works even after kernel restart)
MODULEA_OUTPUT_PATH = globals().get('OUTPUT_PATH', DATA_DIR / 'euro500_netpayout.parquet')

if not MODULEA_OUTPUT_PATH.exists():
    raise FileNotFoundError(f'Missing file: {MODULEA_OUTPUT_PATH}')

m = pd.read_parquet(MODULEA_OUTPUT_PATH).copy()
for c in ['date']:
    if c in m.columns:
        m[c] = pd.to_datetime(m[c], errors='coerce')

targets = [c for c in ['BE', 'assets', 'debt'] if c in m.columns]
if not targets:
    raise KeyError('No Module A columns found (expected BE/assets/debt). Run Step 5 first.')

for c in targets:
    m[c] = pd.to_numeric(m[c], errors='coerce')

if 'date' in m.columns and m['date'].notna().any():
    m['year'] = m['date'].dt.year
else:
    m['year'] = pd.NA

m['all3_available'] = m[targets].notna().all(axis=1)

# Overall coverage
overall = {'rows_total': len(m)}
for c in targets:
    overall[f'cov_{c}'] = float(m[c].notna().mean()) if len(m) else np.nan
overall['cov_all3'] = float(m['all3_available'].mean()) if len(m) else np.nan

display(pd.DataFrame([overall]))

# Year coverage plot (all three items)
if m['year'].notna().any():
    y = m.groupby('year', as_index=False).agg(rows=('year', 'size'))
    for c in targets:
        y[f'cov_{c}'] = m.groupby('year')[c].apply(lambda s: s.notna().mean()).values
    y['cov_all3'] = m.groupby('year')['all3_available'].apply(lambda s: float(s.mean())).values

    plt.figure(figsize=(11, 5))
    for c in targets:
        plt.plot(y['year'], y[f'cov_{c}'], marker='o', linewidth=1.6, label=c)
    plt.title('Module A yearly coverage by item')
    plt.xlabel('Year')
    plt.ylabel('Coverage share')
    plt.ylim(0, 1.02)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


    # Zusatzplot: Module A coverage (Non-Financials only)
    if 'trbc_sector' in m.columns:
        m_nf = m[m['trbc_sector'].astype(str).str.strip().str.lower() != 'financials'].copy()
        if m_nf['year'].notna().any() and not m_nf.empty:
            y_nf = m_nf.groupby('year', as_index=False).agg(rows=('year', 'size'))
            for c in targets:
                y_nf[f'cov_{c}'] = m_nf.groupby('year')[c].apply(lambda s: s.notna().mean()).values
            y_nf['cov_all3'] = m_nf.groupby('year')['all3_available'].apply(lambda s: float(s.mean())).values

            plt.figure(figsize=(11, 5))
            for c in targets:
                plt.plot(y_nf['year'], y_nf[f'cov_{c}'], marker='o', linewidth=1.6, label=c)
            plt.title('Module A yearly coverage by item (Non-Financials)')
            plt.xlabel('Year')
            plt.ylabel('Coverage share')
            plt.ylim(0, 1.02)
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.show()
        else:
            print('No valid Non-Financials rows/year information for Module A coverage plot.')
    else:
        print('Column trbc_sector missing; skipping Module A Non-Financials coverage plot.')
else:
    print('No valid year information available for yearly coverage plot.')




## 3. Modul B: Annual Income Statement (FY)

In [ ]:
# ------------------------------------------------------------
# Step 6 — Module B: Income Statement (FY): Sales, NetIncome, GrossProfit + beta
#   Input  : euro500.parquet
#   Output : euro500_income_statement.parquet
# ------------------------------------------------------------

# -------------------------
# Config
# -------------------------
BASE_PATH = DATA_DIR / 'euro500.parquet'
OUTPUT_PATH_COMBINED = DATA_DIR / 'euro500_netpayout.parquet'

CACHE_DIR_B = CACHE_DATA_DIR / 'moduleB_cache_by_company_id'
CACHE_DIR_B.mkdir(parents=True, exist_ok=True)
BAD_LOG_PATH_B = CACHE_DATA_DIR / 'moduleB_bad_ids.csv'
STEP6_ROWS_PATH = CACHE_DATA_DIR / 'moduleB_step6_rows.parquet'
STEP6_CKPT_PATH = CACHE_DATA_DIR / 'moduleB_step6_checkpoint.json'


TARGET_FIELDS_B = {
    'Sales': 'TR.F.TotRevenue(Period=FY0)',
    'NetIncome': 'TR.F.NetIncAfterTax(Period=FY0)',
    'GrossProfit': 'TR.F.GrossProfIndPropTot(Period=FY0)',
    'Cogs': 'TR.F.COGSTot(Period=FY0)',
}
TARGET_COLS_B = list(TARGET_FIELDS_B.keys())
REQUIRED_COLS_B = ['Sales', 'NetIncome', 'GrossProfit']

ASOF_TOL_DAYS_B = 365
MAX_RETRIES_B = 2
BASE_SLEEP_B = 1.0
FORCE_REFRESH_B = False
CACHE_ONLY_B = True  # False => allow LSEG pull (cache still used)
CACHE_VERSION_B = 'v1'

BATCH_SIZE_B = 100
BATCH_PAUSE_SEC_B = 0
MULTI_UNIVERSE_CHUNK_B = 25

RATE_LIMIT_EVENTS_B = 0
LAST_GETDATA_ERR_MSG_B = ''
DEBUG_VERBOSE_B = False

# LSEG request governor (Step-5 style)
MIN_CALL_INTERVAL_SEC_B = 0.7
RATE_LIMIT_COOLDOWN_SEC_B = 5.0    # wait 5s on first 429 (was 3s)
RATE_LIMIT_MULTIPLIER_B = 2.0      # double each retry (was 1.5)
RATE_LIMIT_HARD_PAUSE_SEC_B = 30.0 # hard pause after repeated 429s (was 10s)
RATE_LIMIT_STREAK_TRIGGER_B = 2
FAIL_FAST_ON_RATE_LIMIT_B = False  # allow retry with cooldown (429 should not abort)
LAST_CALL_TS_B = 0.0
GLOBAL_COOLDOWN_UNTIL_B = 0.0
RATE_LIMIT_STREAK_B = 0

if not BASE_PATH.exists():
    raise FileNotFoundError(f'Missing file: {BASE_PATH}')


# -------------------------
# Helpers
# -------------------------
def _b_clean_str(s: pd.Series) -> pd.Series:
    x = s.astype('string').str.strip()
    return x.where(x.notna() & (x != ''), pd.NA)


def _b_resolve_asof(df: pd.DataFrame) -> pd.Series:
    if 'date' not in df.columns:
        raise ValueError('Missing required date column.')
    return pd.to_datetime(df['date'], errors='coerce').dt.normalize()


def _b_flatten_cols(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    if isinstance(x.columns, pd.MultiIndex):
        x.columns = [' | '.join([str(v) for v in tup if v is not None]).strip() for tup in x.columns]
    else:
        x.columns = [str(c).strip() for c in x.columns]
    return x


def _b_build_company_candidates(company_req: pd.DataFrame) -> list[tuple[str, str]]:
    """Collect all unique identifiers for one company across time."""
    q = company_req.copy().sort_values('date')

    out = []
    seen = set()

    def _norm_isin(value):
        if pd.isna(value):
            return ''
        v = str(value).strip()
        if not v:
            return ''
        if v.upper().startswith('ISIN:'):
            v = v.split(':', 1)[1].strip()
        return v

    def _add(id_type: str, value):
        if pd.isna(value):
            return
        v = str(value).strip()
        if not v:
            return
        key = (id_type, v)
        if key in seen:
            return
        seen.add(key)
        out.append(key)

    if 'ISIN' in q.columns:
        for val in q['ISIN']:
            isin = _norm_isin(val)
            if isin:
                _add('ISIN', isin)

    if 'RIC_current' in q.columns:
        for val in q['RIC_current']:
            _add('RIC', val)

    if 'RIC' in q.columns:
        for val in q['RIC']:
            _add('RIC', val)

    if 'pull_id' in q.columns and 'id_type' in q.columns:
        for id_type, pull_id in zip(q['id_type'], q['pull_id']):
            it = str(id_type).upper().strip() if pd.notna(id_type) else ''
            if it == 'ISIN':
                isin = _norm_isin(pull_id)
                if isin:
                    _add('ISIN', isin)
            elif it == 'RIC':
                _add('RIC', pull_id)

    return out


def _b_cache_file(firm_id: str, id_type: str, pull_id: str) -> Path:
    raw = f'{firm_id}|{id_type}|{pull_id}'
    h = hashlib.sha1(raw.encode('utf-8')).hexdigest()[:12]
    clean = re.sub(r'[^A-Za-z0-9._-]', '_', firm_id)
    preferred = CACHE_DIR_B / f"{clean[:70]}__{id_type}_{h}__{CACHE_VERSION_B}.parquet"
    if preferred.exists():
        return preferred

    matches = sorted(CACHE_DIR_B.glob(f"*__{id_type}_{h}__{CACHE_VERSION_B}.parquet"))
    if matches:
        return matches[0]

    legacy = sorted(CACHE_DIR_B.glob(f"*__{id_type}_{h}.parquet"))
    if legacy:
        return legacy[0]

    return preferred


def _b_load_cache(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=['date'] + TARGET_COLS_B)
    d = pd.read_parquet(path).copy()
    d = _b_flatten_cols(d)
    if 'date' in d.columns:
        d['date'] = pd.to_datetime(d['date'], errors='coerce').dt.normalize()
    else:
        d['date'] = pd.NaT

    for c in TARGET_COLS_B:
        if c not in d.columns:
            d[c] = np.nan
        d[c] = pd.to_numeric(d[c], errors='coerce')

    d = d.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return d[['date'] + TARGET_COLS_B]


def _b_save_cache(path: Path, d: pd.DataFrame) -> None:
    tmp = path.with_suffix(path.suffix + '.tmp')
    out = d.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce').dt.normalize()
    for c in TARGET_COLS_B:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors='coerce')
    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    out.to_parquet(tmp, index=False)
    tmp.replace(path)


def _b_cache_covers_range(cached: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> bool:
    if cached is None or cached.empty:
        return False
    cmin = cached['date'].min()
    cmax = cached['date'].max()
    if pd.isna(cmin) or pd.isna(cmax):
        return False

    has_any_cached_value = bool(cached[REQUIRED_COLS_B].notna().any().any())
    if not has_any_cached_value:
        return False

    return bool((cmin <= start) and (cmax >= end))


def _b_combine(parts: list[pd.DataFrame]) -> pd.DataFrame:
    recs = []
    for part in parts:
        if part is None or len(part) == 0:
            continue
        g = _b_flatten_cols(part.copy())
        for c in ['date'] + TARGET_COLS_B:
            if c not in g.columns:
                g[c] = np.nan
        g['date'] = pd.to_datetime(g['date'], errors='coerce').dt.normalize()
        for c in TARGET_COLS_B:
            g[c] = pd.to_numeric(g[c], errors='coerce')
        g = g[['date'] + TARGET_COLS_B]
        recs.extend(g.to_dict('records'))

    if not recs:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_B)

    out = pd.DataFrame.from_records(recs, columns=['date'] + TARGET_COLS_B)
    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return out


def _b_extract_targets_history(raw: pd.DataFrame) -> pd.DataFrame:
    if raw is None or len(raw) == 0:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_B)

    x = _b_flatten_cols(pd.DataFrame(raw).copy().reset_index())
    if x.empty:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_B)

    date_col = None
    for c in x.columns:
        uc = str(c).upper()
        if 'PERIOD' in uc and 'DATE' in uc:
            date_col = c
            break
    if date_col is None:
        for c in x.columns:
            if 'date' in str(c).lower():
                date_col = c
                break
    if date_col is None:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_B)

    out = pd.DataFrame({'date': pd.to_datetime(x[date_col], errors='coerce').dt.normalize()})

    def _norm(txt: str) -> str:
        return re.sub(r'[^A-Z0-9]', '', str(txt).upper())

    def _field_token(field_expr: str) -> str:
        m = re.search(r'TR\.F\.([A-Z0-9_]+)', str(field_expr), flags=re.IGNORECASE)
        return _norm(m.group(1)) if m else _norm(field_expr)

    fallback_tokens = {
        'Sales': ['TOTREVENUE', 'REVENUE', 'SALES'],
        'NetIncome': ['NETINCAFTERTAX', 'NETINCOME', 'NETPROFIT'],
        'GrossProfit': ['GROSSPROFINDPROPTOT', 'GROSSPROFIT'],
        'Cogs': ['COGSTOT', 'COGS', 'COSTOFGOODSSOLD'],
    }

    norm_cols = {c: _norm(c) for c in x.columns}

    for tgt, field_expr in TARGET_FIELDS_B.items():
        token = _field_token(field_expr)
        cand = None
        for c, cn in norm_cols.items():
            if token and (token in cn or cn in token):
                cand = c
                break
        if cand is None:
            for alt in fallback_tokens.get(tgt, []):
                alt_n = _norm(alt)
                for c, cn in norm_cols.items():
                    if alt_n and alt_n in cn:
                        cand = c
                        break
                if cand is not None:
                    break
        out[tgt] = pd.to_numeric(x[cand], errors='coerce') if cand is not None else np.nan

    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return out[['date'] + TARGET_COLS_B]


def _b_govern_requests() -> None:
    global LAST_CALL_TS_B, GLOBAL_COOLDOWN_UNTIL_B
    now = time.time()
    if now < GLOBAL_COOLDOWN_UNTIL_B:
        wait = GLOBAL_COOLDOWN_UNTIL_B - now
        if wait > 0:
            time.sleep(wait)
    now = time.time()
    gap = now - LAST_CALL_TS_B
    if gap < MIN_CALL_INTERVAL_SEC_B:
        time.sleep(MIN_CALL_INTERVAL_SEC_B - gap)


def _b_call_get_data(universe: list[str], fields: list[str], params: dict, dbg_label: str) -> pd.DataFrame:
    global LAST_CALL_TS_B, GLOBAL_COOLDOWN_UNTIL_B, RATE_LIMIT_STREAK_B, RATE_LIMIT_EVENTS_B, LAST_GETDATA_ERR_MSG_B

    LAST_GETDATA_ERR_MSG_B = ''
    last_err = None
    raw = None
    for r in range(MAX_RETRIES_B):
        try:
            _b_govern_requests()
            raw = ld.get_data(universe=universe, fields=fields, parameters=params)
            LAST_CALL_TS_B = time.time()
            RATE_LIMIT_STREAK_B = 0
            LAST_GETDATA_ERR_MSG_B = ''
            break
        except Exception as e:
            LAST_CALL_TS_B = time.time()
            last_err = e
            LAST_GETDATA_ERR_MSG_B = str(e)
            msg = str(e)
            is_rate_limit = ('Too many requests' in msg) or ('429' in msg)
            if is_rate_limit:
                RATE_LIMIT_STREAK_B += 1
                RATE_LIMIT_EVENTS_B += 1
                cooldown = RATE_LIMIT_COOLDOWN_SEC_B * (RATE_LIMIT_MULTIPLIER_B ** r) + random.random()
                if RATE_LIMIT_STREAK_B >= RATE_LIMIT_STREAK_TRIGGER_B:
                    cooldown = max(cooldown, RATE_LIMIT_HARD_PAUSE_SEC_B)
                GLOBAL_COOLDOWN_UNTIL_B = max(GLOBAL_COOLDOWN_UNTIL_B, time.time() + cooldown)
                if DEBUG_VERBOSE_B:
                    print(
                        f'[DEBUG rate limit B] id={dbg_label} retry={r+1}/{MAX_RETRIES_B} '
                        f'streak={RATE_LIMIT_STREAK_B} cooldown={cooldown:.1f}s '
                        f'err={type(e).__name__}: {e}'
                    )
                if FAIL_FAST_ON_RATE_LIMIT_B:
                    break
                time.sleep(cooldown)
            else:
                RATE_LIMIT_STREAK_B = 0
                if DEBUG_VERBOSE_B:
                    print(f'[DEBUG get_data B error] id={dbg_label} retry={r+1}/{MAX_RETRIES_B} err={type(e).__name__}: {e}')
                time.sleep(BASE_SLEEP_B * (2 ** r) + random.random() * 0.3)

    if raw is None or len(raw) == 0:
        if last_err is not None:
            msg = str(last_err)
            if 'Unable to resolve all requested identifiers' not in msg:
                print(f'[WARN get_data B failed] id={dbg_label}: {type(last_err).__name__}: {last_err}')
        elif DEBUG_VERBOSE_B:
            print(f'[DEBUG empty raw B no exception] id={dbg_label}')
        return pd.DataFrame()

    return pd.DataFrame(raw)


def _b_pull_targets_multi_segment(pull_ids: list[str], start: pd.Timestamp, end: pd.Timestamp) -> dict[str, pd.DataFrame]:
    if not pull_ids or pd.isna(start) or pd.isna(end) or start > end:
        return {}

    uniq = []
    seen = set()
    for pid in pull_ids:
        v = str(pid).strip()
        if not v:
            continue
        k = v.upper()
        if k in seen:
            continue
        seen.add(k)
        uniq.append(v)

    if not uniq:
        return {}

    params = {
        'SDate': start.strftime('%Y-%m-%d'),
        'EDate': end.strftime('%Y-%m-%d'),
        'Frq': 'FY',
    }
    fields = ['TR.F.PeriodEndDate(Period=FY0)'] + list(TARGET_FIELDS_B.values())

    out = {pid: pd.DataFrame(columns=['date'] + TARGET_COLS_B) for pid in uniq}

    def _assign_from_raw(ids: list[str], raw_df: pd.DataFrame) -> None:
        if raw_df is None or raw_df.empty:
            return
        x = _b_flatten_cols(raw_df)
        if x.empty:
            return

        inst_col = None
        for c in x.columns:
            cl = c.lower()
            if 'instrument' in cl or cl in {'ric', 'isin'}:
                inst_col = c
                break

        if inst_col is None:
            # No instrument column — attribute whole frame to the single identifier.
            if len(ids) == 1:
                hist = _b_extract_targets_history(x)
                if not hist.empty:
                    out[ids[0]] = _b_combine([out[ids[0]], hist])
            return

        inst_norm = x[inst_col].astype('string').str.strip().str.upper()
        for pid in ids:
            pnorm = str(pid).strip().upper()
            g = x.loc[inst_norm == pnorm].copy()
            if g.empty:
                # Some providers return decorated instruments (e.g. LSEG strips ISIN: prefix).
                g = x.loc[inst_norm.str.contains(re.escape(pnorm), regex=True, na=False)].copy()
            if g.empty:
                # If single-id chunk and instrument mismatch, still use full frame.
                if len(ids) == 1:
                    g = x.copy()
                else:
                    continue
            hist = _b_extract_targets_history(g)
            if not hist.empty:
                out[pid] = _b_combine([out[pid], hist])

    def _periodend_collect_error(msg: str) -> bool:
        m = (msg or '').lower()
        return ('unable to collect data for the field' in m) and ('periodenddate' in m)

    def _fetch_ids(ids: list[str]) -> None:
        if not ids:
            return
        label = f'multiB[{len(ids)}]:' + ','.join(ids[:3])
        raw_df = _b_call_get_data(universe=ids, fields=fields, params=params, dbg_label=label)

        if raw_df.empty:
            err = LAST_GETDATA_ERR_MSG_B
            # Same strategy as Step 5: split poisoned multi requests recursively.
            if len(ids) > 1 and _periodend_collect_error(err):
                mid = len(ids) // 2
                _fetch_ids(ids[:mid])
                _fetch_ids(ids[mid:])
            return

        _assign_from_raw(ids, raw_df)

    for i in range(0, len(uniq), MULTI_UNIVERSE_CHUNK_B):
        chunk = uniq[i:i + MULTI_UNIVERSE_CHUNK_B]
        _fetch_ids(chunk)

    return out


def _b_update_company_cache(firm_id: str, id_type: str, pull_id: str, start: pd.Timestamp, end: pd.Timestamp, force_refresh: bool = False) -> pd.DataFrame:
    path = _b_cache_file(firm_id, id_type, pull_id)
    cached = pd.DataFrame(columns=['date'] + TARGET_COLS_B) if force_refresh else _b_load_cache(path)

    if _b_cache_covers_range(cached, start, end) and (not force_refresh):
        return cached

    if CACHE_ONLY_B:
        return cached

    pulled = _b_pull_targets_multi_segment([pull_id], start, end).get(pull_id, pd.DataFrame(columns=['date'] + TARGET_COLS_B))
    combined = _b_combine([cached, pulled])
    if not combined.empty or force_refresh:
        _b_save_cache(path, combined)
    return _b_load_cache(path)


def _b_map_to_asof(req_dates: pd.Series, hist: pd.DataFrame, tol_days: int) -> pd.DataFrame:
    left = pd.DataFrame({'date': pd.to_datetime(req_dates, errors='coerce').dt.normalize()}).dropna().sort_values('date')
    if left.empty:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_B)

    if hist is None or hist.empty:
        for c in TARGET_COLS_B:
            left[c] = np.nan
        return left

    right = hist[['date'] + TARGET_COLS_B].copy().sort_values('date').drop_duplicates(['date'], keep='last')
    out = pd.merge_asof(
        left,
        right,
        on='date',
        direction='backward',
        tolerance=pd.Timedelta(days=tol_days),
    )
    return out



# -------------------------
# Build request table (company x asof) + dynamic fallback candidates
# -------------------------
base = pd.read_parquet(BASE_PATH).copy()
base['date'] = _b_resolve_asof(base)

for c in ['firm_id', 'ISIN', 'RIC_current', 'RIC', 'pull_id', 'id_type', 'firm_id']:
    if c in base.columns:
        base[c] = _b_clean_str(base[c])

if 'id_type' not in base.columns:
    base['id_type'] = np.select(
        [base.get('ISIN', pd.Series(pd.NA, index=base.index)).notna(),
         base.get('RIC_current', pd.Series(pd.NA, index=base.index)).notna(),
         base.get('RIC', pd.Series(pd.NA, index=base.index)).notna()],
        ['ISIN', 'RIC', 'RIC'],
        default=pd.NA,
    )
if 'pull_id' not in base.columns:
    base['pull_id'] = np.select(
        [base.get('ISIN', pd.Series(pd.NA, index=base.index)).notna(),
         base.get('RIC_current', pd.Series(pd.NA, index=base.index)).notna(),
         base.get('RIC', pd.Series(pd.NA, index=base.index)).notna()],
        [base.get('ISIN', pd.Series(pd.NA, index=base.index)),
         base.get('RIC_current', pd.Series(pd.NA, index=base.index)),
         base.get('RIC', pd.Series(pd.NA, index=base.index))],
        default=pd.NA,
    )
if 'firm_id' not in base.columns:
    base['firm_id'] = _b_clean_str(base['id_type']).astype('string') + ':' + _b_clean_str(base['pull_id']).astype('string')
    base.loc[base['pull_id'].isna(), 'firm_id'] = pd.NA

req_cols_b = [c for c in ['firm_id', 'date', 'ISIN', 'RIC_current', 'RIC', 'id_type', 'pull_id'] if c in base.columns]
req_b = (
    base[req_cols_b]
    .dropna(subset=['firm_id', 'date'])
    .drop_duplicates(['firm_id', 'date'], keep='last')
    .reset_index(drop=True)
)
if req_b.empty:
    raise ValueError('No valid (firm_id, date) rows found for Step 6.')

company_candidates_map_b = {}
for ck, g in req_b.groupby('firm_id', sort=False):
    company_candidates_map_b[str(ck)] = _b_build_company_candidates(g)

req_b['n_id_candidates'] = req_b['firm_id'].astype(str).map(lambda ck: len(company_candidates_map_b.get(ck, [])))

print('Step 6 request rows (company x asof):', len(req_b))
print('As-of range:', req_b['date'].min(), 'to', req_b['date'].max())
print('Unique companies:', req_b['firm_id'].nunique())
print('Active LSEG fields:', TARGET_FIELDS_B)
print('Mode:', 'CACHE_ONLY' if CACHE_ONLY_B else 'CACHE+NETWORK')
print('ID fallback order (historical per firm): all ISINs -> all RIC_current -> all RIC (+ pull_id/id_type history)')

# -------------------------
# Pull loop (batched + resume)
# -------------------------
companies_all_b = req_b['firm_id'].dropna().unique().tolist()
N_total_b = len(companies_all_b)

existing_step_rows_b = pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS_B)
if STEP6_ROWS_PATH.exists():
    try:
        existing_step_rows_b = pd.read_parquet(STEP6_ROWS_PATH)
        if 'date' in existing_step_rows_b.columns:
            existing_step_rows_b['date'] = pd.to_datetime(existing_step_rows_b['date'], errors='coerce').dt.normalize()
        for c in TARGET_COLS_B:
            if c not in existing_step_rows_b.columns:
                existing_step_rows_b[c] = np.nan
            existing_step_rows_b[c] = pd.to_numeric(existing_step_rows_b[c], errors='coerce')
        existing_step_rows_b = existing_step_rows_b[['firm_id', 'date'] + TARGET_COLS_B].dropna(subset=['firm_id', 'date'])
        existing_step_rows_b = existing_step_rows_b.drop_duplicates(['firm_id', 'date'], keep='last')
    except Exception as e:
        print(f'Warning: failed to read STEP6 rows cache, continuing empty: {e}')
        existing_step_rows_b = pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS_B)

processed_from_rows_b = set(existing_step_rows_b['firm_id'].dropna().astype(str).unique().tolist()) if not existing_step_rows_b.empty else set()
processed_from_ckpt_b = set()
if STEP6_CKPT_PATH.exists():
    try:
        ck = json.loads(STEP6_CKPT_PATH.read_text())
        processed_from_ckpt_b = set(str(x) for x in ck.get('processed_companies', []) if str(x).strip())
    except Exception as e:
        print(f'Warning: failed to read checkpoint, ignoring: {e}')

cache_file_count_b = len(list(CACHE_DIR_B.glob('*.parquet')))
if cache_file_count_b == 0 and (processed_from_rows_b or processed_from_ckpt_b):
    print('Step6: cache directory is empty -> ignoring step rows/checkpoint and starting full rebuild.')
    processed_from_rows_b = set()
    processed_from_ckpt_b = set()

processed_companies_b = set(processed_from_rows_b) | set(processed_from_ckpt_b)
companies_b = [c for c in companies_all_b if str(c) not in processed_companies_b]
N_b = len(companies_b)

cache_firm_ids_b = set()
for _p in CACHE_DIR_B.glob('*.parquet'):
    _name = _p.name
    if '__' in _name:
        cache_firm_ids_b.add(_name.split('__', 1)[0])
print('Resume info: total_companies=', N_total_b, 'already_processed=', len(processed_companies_b), 'remaining=', N_b, 'cache_files=', cache_file_count_b, 'cache_firms=', len(cache_firm_ids_b))

run_t0_b = time.time()
new_rows_out_b = []
bad_rows_b = []

total_cand_calls_b = 0
total_all3_resolved_b = 0
total_not_all3_b = 0
total_item_found_b = {c: 0 for c in REQUIRED_COLS_B}

if not CACHE_ONLY_B:
    ld.open_session()
try:
    if N_b == 0:
        print('No remaining companies to pull in Step 6.')

    n_batches_b = int(np.ceil(N_b / BATCH_SIZE_B)) if N_b > 0 else 0
    for b_ix, b_start in enumerate(range(0, N_b, BATCH_SIZE_B), start=1):
        b_end = min(N_b, b_start + BATCH_SIZE_B)
        batch_companies = companies_b[b_start:b_end]
        batch_t0 = time.time()
        batch_new_rows = []
        batch_processed = []

        print(f'[BATCH {b_ix}/{n_batches_b}] companies={len(batch_companies)} idx={b_start+1}-{b_end}')

        for k, firm_id in enumerate(batch_companies, start=1):
            q = req_b[req_b['firm_id'] == firm_id].copy().sort_values('date')
            dates = q['date'].dropna().drop_duplicates().sort_values().reset_index(drop=True)
            if dates.empty:
                continue

            start = pd.to_datetime(dates.min()).normalize()
            end = pd.to_datetime(dates.max()).normalize()

            panel = pd.DataFrame({'date': dates})
            for c in TARGET_COLS_B:
                panel[c] = np.nan

            cands = company_candidates_map_b.get(str(firm_id), [])
            cand_used = 0
            attempted_ids = []

            for cand in cands:
                if panel[REQUIRED_COLS_B].notna().all(axis=1).all():
                    break
                if (not isinstance(cand, (list, tuple))) or len(cand) != 2:
                    continue

                id_type = str(cand[0]).upper().strip()
                pull_id = str(cand[1]).strip()
                if not id_type or not pull_id:
                    continue

                cand_used += 1
                total_cand_calls_b += 1
                attempted_ids.append(f'{id_type}:{pull_id}')

                cache_path = _b_cache_file(str(firm_id), id_type, pull_id)
                cached_pre = _b_load_cache(cache_path)
                if _b_cache_covers_range(cached_pre, start, end) and (not FORCE_REFRESH_B):
                    hist = cached_pre
                else:
                    hist = _b_update_company_cache(
                        firm_id=str(firm_id),
                        id_type=id_type,
                        pull_id=pull_id,
                        start=start,
                        end=end,
                        force_refresh=FORCE_REFRESH_B,
                    )
                mapped = _b_map_to_asof(panel['date'], hist, tol_days=ASOF_TOL_DAYS_B)
                if mapped.empty:
                    continue

                for c in TARGET_COLS_B:
                    cur = pd.to_numeric(panel[c], errors='coerce')
                    nxt = pd.to_numeric(mapped.get(c, np.nan), errors='coerce')
                    panel[c] = cur.where(cur.notna(), nxt)

            # GrossProfit fallback: TotRevenue - Cogs
            sales_s = pd.to_numeric(panel['Sales'], errors='coerce')
            gp_s = pd.to_numeric(panel['GrossProfit'], errors='coerce')
            cogs_s = pd.to_numeric(panel['Cogs'], errors='coerce')
            panel['GrossProfit'] = gp_s.combine_first(sales_s - cogs_s)

            panel.insert(0, 'firm_id', firm_id)
            recs = panel[['firm_id', 'date'] + TARGET_COLS_B].to_dict('records')
            new_rows_out_b.extend(recs)
            batch_new_rows.extend(recs)
            batch_processed.append(str(firm_id))

            all3_resolved = int(panel[REQUIRED_COLS_B].notna().all(axis=1).sum())
            not_all3 = int(len(panel) - all3_resolved)
            item_found = {c: int(panel[c].notna().sum()) for c in REQUIRED_COLS_B}
            total_all3_resolved_b += all3_resolved
            total_not_all3_b += not_all3
            for c in REQUIRED_COLS_B:
                total_item_found_b[c] += item_found[c]

            any_required_found = bool(panel[REQUIRED_COLS_B].notna().any().any())
            if (not any_required_found) and attempted_ids:
                print(
                    f'[WARN company identifiers unresolved] company={str(firm_id)[:40]} '
                    f'ids={attempted_ids[:4]}'
                )

            unresolved = panel[panel[REQUIRED_COLS_B].isna().any(axis=1)]
            if not unresolved.empty:
                bad_rows_b.extend(
                    unresolved[['firm_id', 'date']]
                    .assign(reason='not_all3_resolved', n_candidates=int(len(cands)))
                    .to_dict('records')
                )

            elapsed = time.time() - run_t0_b
            print(
                f'[BATCH {b_ix}/{n_batches_b}] [{b_start+k}/{N_b}] company={str(firm_id)[:40]} rows={len(dates)} '
                f'cand_used={cand_used}/{len(cands)} all3_resolved={all3_resolved} not_all3={not_all3} '
                f'found_Sales={item_found.get("Sales",0)} found_NetIncome={item_found.get("NetIncome",0)} found_GrossProfit={item_found.get("GrossProfit",0)} '
                f'elapsed={elapsed/60:.1f}m'
            )

        # Persist batch rows for resume
        if batch_new_rows:
            batch_df = pd.DataFrame(batch_new_rows)
            batch_df['date'] = pd.to_datetime(batch_df['date'], errors='coerce').dt.normalize()
            for c in TARGET_COLS_B:
                batch_df[c] = pd.to_numeric(batch_df[c], errors='coerce')
            batch_df = batch_df.dropna(subset=['firm_id', 'date'])

            if STEP6_ROWS_PATH.exists():
                prev = pd.read_parquet(STEP6_ROWS_PATH)
                if prev is None or prev.empty:
                    combined = batch_df.copy()
                else:
                    prev = prev.copy()
                    prev['date'] = pd.to_datetime(prev.get('date'), errors='coerce').dt.normalize()
                    for c in TARGET_COLS_B:
                        if c not in prev.columns:
                            prev[c] = np.nan
                        prev[c] = pd.to_numeric(prev[c], errors='coerce')
                    prev = prev[['firm_id', 'date'] + TARGET_COLS_B].dropna(subset=['firm_id', 'date'])
                    combined = pd.concat([prev, batch_df], ignore_index=True)
            else:
                combined = batch_df.copy()

            combined = combined.sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last')
            combined.to_parquet(STEP6_ROWS_PATH, index=False)

        # Persist checkpoint
        processed_companies_b.update(batch_processed)
        ckpt_payload = {
            'processed_companies': sorted(processed_companies_b),
            'last_batch': b_ix,
            'remaining_companies': max(0, N_total_b - len(processed_companies_b)),
            'updated_at_utc': pd.Timestamp.utcnow().isoformat(),
        }
        STEP6_CKPT_PATH.write_text(json.dumps(ckpt_payload, ensure_ascii=False, indent=2))

        batch_elapsed = (time.time() - batch_t0) / 60
        print(f'[BATCH {b_ix}/{n_batches_b} DONE] processed={len(batch_processed)} batch_elapsed={batch_elapsed:.1f}m')

        if b_end < N_b and BATCH_PAUSE_SEC_B > 0:
            print(f'[BATCH PAUSE] sleeping {BATCH_PAUSE_SEC_B:.0f}s before next batch...')
            time.sleep(BATCH_PAUSE_SEC_B)
finally:
    if not CACHE_ONLY_B:
        ld.close_session()

print(
    f'Done Step 6: total_companies={N_total_b}, run_remaining_start={N_b}, candidate_calls={total_cand_calls_b}, '
    f'all3_resolved_rows={total_all3_resolved_b}, not_all3_rows={total_not_all3_b}, '
    f'found_Sales={total_item_found_b.get("Sales",0)}, found_NetIncome={total_item_found_b.get("NetIncome",0)}, found_GrossProfit={total_item_found_b.get("GrossProfit",0)}'
)

rebuild_output_b = FORCE_REFRESH_B or (N_b > 0) or (not STEP6_ROWS_PATH.exists()) or (not OUTPUT_PATH_COMBINED.exists())
if (not rebuild_output_b) and BASE_PATH.exists() and OUTPUT_PATH_COMBINED.exists():
    rebuild_output_b = BASE_PATH.stat().st_mtime > OUTPUT_PATH_COMBINED.stat().st_mtime

# Force rebuild if combined output does not yet contain Module-B columns.
if (not rebuild_output_b) and OUTPUT_PATH_COMBINED.exists():
    try:
        chk = pd.read_parquet(OUTPUT_PATH_COMBINED)
        needed_b = {'Sales', 'NetIncome', 'GrossProfit'}
        if not needed_b.issubset(set(chk.columns)):
            rebuild_output_b = True
            print('Step 6 rebuild forced: combined output missing Module-B columns.')
    except Exception:
        rebuild_output_b = True
        print('Step 6 rebuild forced: unable to inspect combined output file.')

# -------------------------
# Build final output table
# -------------------------
if rebuild_output_b and STEP6_ROWS_PATH.exists():
    out_panel_b = pd.read_parquet(STEP6_ROWS_PATH)
else:
    out_panel_b = pd.DataFrame(new_rows_out_b)

if rebuild_output_b and out_panel_b.empty:
    out_panel_b = pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS_B)
elif rebuild_output_b:
    out_panel_b['date'] = pd.to_datetime(out_panel_b['date'], errors='coerce').dt.normalize()
    for c in TARGET_COLS_B:
        if c not in out_panel_b.columns:
            out_panel_b[c] = np.nan
        out_panel_b[c] = pd.to_numeric(out_panel_b[c], errors='coerce')

    # Recompute fallback after resume load
    out_panel_b['GrossProfit'] = pd.to_numeric(out_panel_b['GrossProfit'], errors='coerce').combine_first(
        pd.to_numeric(out_panel_b['Sales'], errors='coerce') - pd.to_numeric(out_panel_b['Cogs'], errors='coerce')
    )

    out_panel_b = out_panel_b[['firm_id', 'date'] + TARGET_COLS_B]
    out_panel_b = out_panel_b.sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last')

if 'base' not in locals():
    base = pd.read_parquet(BASE_PATH).copy()
    base['date'] = _b_resolve_asof(base)

if rebuild_output_b:
    # Stage 1: preferred merge on firm_id + date
    if 'firm_id' in base.columns:
        out_panel_b_firm = out_panel_b.rename(columns={'firm_id': 'firm_id'})
        out_b = base.merge(out_panel_b_firm, on=['firm_id', 'date'], how='left', suffixes=('', '_firm'))
    else:
        out_b = base.copy()

    # Stage 2 fallback: merge on firm_id + date
    add_b = base.merge(out_panel_b, on=['firm_id', 'date'], how='left', suffixes=('', '_key'))

    for c in TARGET_COLS_B:
        old_s = pd.to_numeric(out_b[c], errors='coerce') if c in out_b.columns else pd.Series(np.nan, index=out_b.index)
        new_firm_s = pd.to_numeric(out_b[f'{c}_firm'] if f'{c}_firm' in out_b.columns else pd.Series(np.nan, index=out_b.index), errors='coerce')
        new_key_s = pd.to_numeric(add_b[f'{c}_key'] if f'{c}_key' in add_b.columns else pd.Series(np.nan, index=out_b.index), errors='coerce')
        out_b[c] = new_firm_s.combine_first(new_key_s).combine_first(old_s)
        if f'{c}_firm' in out_b.columns:
            out_b = out_b.drop(columns=[f'{c}_firm'])

    # Ensure fallback in final output
    out_b['GrossProfit'] = pd.to_numeric(out_b['GrossProfit'], errors='coerce').combine_first(
        pd.to_numeric(out_b['Sales'], errors='coerce') - pd.to_numeric(out_b['Cogs'], errors='coerce')
    )


    # Keep requested outputs + beta; Cogs stays internal only
    if 'Cogs' in out_b.columns:
        out_b = out_b.drop(columns=['Cogs'])

    # Bring Module-A fields into the combined table.
    # Priority:
    # 1) existing combined output (if it already contains Module-A cols),
    # 2) base euro500 panel as fallback.
    moduleA_cols = []
    modA_all = None
    if OUTPUT_PATH_COMBINED.exists():
        _tmp = pd.read_parquet(OUTPUT_PATH_COMBINED).copy()
        _cols = [c for c in ['BE', 'assets', 'debt'] if c in _tmp.columns]
        if _cols:
            modA_all = _tmp
            moduleA_cols = _cols

    if (modA_all is None) and BASE_PATH.exists():
        _tmp = pd.read_parquet(BASE_PATH).copy()
        _cols = [c for c in ['BE', 'assets', 'debt'] if c in _tmp.columns]
        if _cols:
            modA_all = _tmp
            moduleA_cols = _cols

    if modA_all is not None and moduleA_cols:
        # Keep time axis consistent across modules.
        modA_all['date'] = _b_resolve_asof(modA_all)

        modA = modA_all[['firm_id', 'date'] + moduleA_cols].dropna(subset=['firm_id', 'date'])
        modA = modA.drop_duplicates(['firm_id', 'date'], keep='last')
        out_b = out_b.merge(modA, on=['firm_id', 'date'], how='left', suffixes=('', '_modA'))

        for c in moduleA_cols:
            if f'{c}_modA' in out_b.columns:
                cur = pd.to_numeric(out_b[c], errors='coerce') if c in out_b.columns else pd.Series(np.nan, index=out_b.index)
                old = pd.to_numeric(out_b[f'{c}_modA'], errors='coerce')
                out_b[c] = cur.combine_first(old)
                out_b = out_b.drop(columns=[f'{c}_modA'])

    
    # Keep only canonical beta column in combined output
    if ('beta' not in out_b.columns) and ('beta_3m' in out_b.columns):
        out_b['beta'] = pd.to_numeric(out_b['beta_3m'], errors='coerce')
    if 'beta_3m' in out_b.columns:
        out_b = out_b.drop(columns=['beta_3m'])

    out_b.to_parquet(OUTPUT_PATH_COMBINED, index=False)
    euro500_netpayout_df = out_b.copy()

    print('Saved combined output (Step 5 + 6):', OUTPUT_PATH_COMBINED, 'rows:', len(out_b))
    print('Data Wrangler variable ready: euro500_netpayout_df')
else:
    out_b = pd.read_parquet(OUTPUT_PATH_COMBINED)
    euro500_netpayout_df = out_b.copy()
    print('Skipped Step-6 rebuild (already up-to-date):', OUTPUT_PATH_COMBINED, 'rows:', len(out_b))
    print('Data Wrangler variable ready: euro500_netpayout_df')

if bad_rows_b:
    bad_df = pd.DataFrame(bad_rows_b)
    if BAD_LOG_PATH_B.exists():
        old = pd.read_csv(BAD_LOG_PATH_B)
        for c in ['firm_id', 'date', 'reason', 'n_candidates']:
            if c not in old.columns:
                old[c] = pd.NA
            if c not in bad_df.columns:
                bad_df[c] = pd.NA
        out_bad = pd.DataFrame.from_records(
            old[['firm_id', 'date', 'reason', 'n_candidates']].to_dict('records')
            + bad_df[['firm_id', 'date', 'reason', 'n_candidates']].to_dict('records'),
            columns=['firm_id', 'date', 'reason', 'n_candidates'],
        )
    else:
        out_bad = bad_df

    out_bad['date'] = pd.to_datetime(out_bad['date'], errors='coerce').dt.normalize()
    out_bad = out_bad.drop_duplicates(subset=['firm_id', 'date', 'reason'], keep='last')
    out_bad.to_csv(BAD_LOG_PATH_B, index=False)
    print('Updated Module-B bad-id log:', BAD_LOG_PATH_B, 'rows:', len(out_bad))



### 3B. Coverage-Analyse Modul B (`Sales`, `NetIncome`, `GrossProfit`)

In [ ]:
# Robust path resolution for Step 6B
MODULEB_OUTPUT_PATH = globals().get('OUTPUT_PATH_COMBINED', DATA_DIR / 'euro500_netpayout.parquet')

if not MODULEB_OUTPUT_PATH.exists():
    raise FileNotFoundError(f'Missing file: {MODULEB_OUTPUT_PATH}')

mb = pd.read_parquet(MODULEB_OUTPUT_PATH).copy()
if 'date' in mb.columns:
    mb['date'] = pd.to_datetime(mb['date'], errors='coerce')

targets_b = [c for c in ['Sales', 'NetIncome', 'GrossProfit'] if c in mb.columns]
if not targets_b:
    raise KeyError('No Module B columns found (expected Sales/NetIncome/GrossProfit). Run Step 6 first.')

for c in targets_b:
    mb[c] = pd.to_numeric(mb[c], errors='coerce')

if 'date' in mb.columns and mb['date'].notna().any():
    mb['year'] = mb['date'].dt.year
else:
    mb['year'] = pd.NA

mb['all3_available'] = mb[targets_b].notna().all(axis=1)

overall_b = {'rows_total': len(mb)}
for c in targets_b:
    overall_b[f'cov_{c}'] = float(mb[c].notna().mean()) if len(mb) else np.nan
overall_b['cov_all3'] = float(mb['all3_available'].mean()) if len(mb) else np.nan

display(pd.DataFrame([overall_b]))

if mb['year'].notna().any():
    yb = mb.groupby('year', as_index=False).agg(rows=('year', 'size'))
    for c in targets_b:
        yb[f'cov_{c}'] = mb.groupby('year')[c].apply(lambda s: s.notna().mean()).values
    yb['cov_all3'] = mb.groupby('year')['all3_available'].apply(lambda s: float(s.mean())).values

    plt.figure(figsize=(11, 5))
    for c in targets_b:
        plt.plot(yb['year'], yb[f'cov_{c}'], marker='o', linewidth=1.6, label=c)
    plt.title('Module B yearly coverage by item')
    plt.xlabel('Year')
    plt.ylabel('Coverage share')
    plt.ylim(0, 1.02)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


    # Zusatzplot: Module B coverage (Non-Financials only)
    if 'trbc_sector' in mb.columns:
        mb_nf = mb[mb['trbc_sector'].astype(str).str.strip().str.lower() != 'financials'].copy()
        if mb_nf['year'].notna().any() and not mb_nf.empty:
            yb_nf = mb_nf.groupby('year', as_index=False).agg(rows=('year', 'size'))
            for c in targets_b:
                yb_nf[f'cov_{c}'] = mb_nf.groupby('year')[c].apply(lambda s: s.notna().mean()).values
            yb_nf['cov_all3'] = mb_nf.groupby('year')['all3_available'].apply(lambda s: float(s.mean())).values

            plt.figure(figsize=(11, 5))
            for c in targets_b:
                plt.plot(yb_nf['year'], yb_nf[f'cov_{c}'], marker='o', linewidth=1.6, label=c)
            plt.title('Module B yearly coverage by item (Non-Financials)')
            plt.xlabel('Year')
            plt.ylabel('Coverage share')
            plt.ylim(0, 1.02)
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.show()
        else:
            print('No valid Non-Financials rows/year information for Module B coverage plot.')
    else:
        print('Column trbc_sector missing; skipping Module B Non-Financials coverage plot.')
else:
    print('No valid year information available for yearly coverage plot.')




## 4. Modul C: Annual Cashflow und Payouts (FY)

In [ ]:
# ------------------------------------------------------------
# Step 7 — Module C: Cashflow / Payouts (FY): Dividends + Buybacks
#   Input  : euro500.parquet
#   Output : euro500_netpayout.parquet (combined table)
# ------------------------------------------------------------

# -------------------------
# Config
# -------------------------
BASE_PATH_E = DATA_DIR / 'euro500.parquet'
OUTPUT_PATH_COMBINED = DATA_DIR / 'euro500_netpayout.parquet'

CACHE_DIR_C = CACHE_DATA_DIR / 'moduleC_cache_by_company_id'
CACHE_DIR_C.mkdir(parents=True, exist_ok=True)
BAD_LOG_PATH_C = CACHE_DATA_DIR / 'moduleC_bad_ids.csv'
STEP7_ROWS_PATH = CACHE_DATA_DIR / 'moduleC_step7_rows.parquet'
STEP7_CKPT_PATH = CACHE_DATA_DIR / 'moduleC_step7_checkpoint.json'

TARGET_FIELDS_E = {
    'Dividends': 'TR.F.DivPaidCashTotCF(Period=FY0)',
    'Buybacks': 'TR.F.ComStockBuybackNet(Period=FY0)',
}
TARGET_COLS_E = list(TARGET_FIELDS_E.keys())
REQUIRED_COLS_E = ['Dividends', 'Buybacks']

ASOF_TOL_DAYS_E = 365
MAX_RETRIES_E = 2
BASE_SLEEP_E = 1.0
FORCE_REFRESH_E = False
CACHE_ONLY_E = True  # False => allow LSEG pull (cache still used)
CACHE_VERSION_E = 'v1'

BATCH_SIZE_E = 100
BATCH_PAUSE_SEC_E = 0
MULTI_UNIVERSE_CHUNK_E = 25


if not BASE_PATH_E.exists():
    raise FileNotFoundError(f'Missing file: {BASE_PATH_E}')


# -------------------------
# Helpers
# -------------------------
def _e_clean_str(s: pd.Series) -> pd.Series:
    x = s.astype('string').str.strip()
    return x.where(x.notna() & (x != ''), pd.NA)


def _e_resolve_asof(df: pd.DataFrame) -> pd.Series:
    if 'date' not in df.columns:
        raise ValueError('Missing required date column.')
    return pd.to_datetime(df['date'], errors='coerce').dt.normalize()


def _e_flatten_cols(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    if isinstance(x.columns, pd.MultiIndex):
        x.columns = [' | '.join([str(v) for v in tup if v is not None]).strip() for tup in x.columns]
    else:
        x.columns = [str(c).strip() for c in x.columns]
    return x


def _e_norm_isin(value) -> str | None:
    if pd.isna(value):
        return None
    v = str(value).strip()
    if not v:
        return None
    return v.split(':', 1)[1].strip() if v.upper().startswith('ISIN:') else v


def _e_build_company_candidates(company_req: pd.DataFrame) -> list[tuple[str, str]]:
    """Collect all unique identifiers for one company across time.

    Order: all ISINs, then all RIC_current, then all RIC (chronological first-seen).
    """
    q = company_req.copy().sort_values('date')

    out = []
    seen = set()

    def _add(id_type: str, value):
        if pd.isna(value):
            return
        v = str(value).strip()
        if not v:
            return
        key = (id_type, v)
        if key in seen:
            return
        seen.add(key)
        out.append(key)

    # 1) all ISINs observed over time for this company
    if 'ISIN' in q.columns:
        for val in q['ISIN']:
            norm = _e_norm_isin(val)
            if norm:
                _add('ISIN', norm)
                _add('ISIN', f'ISIN:{norm}')

    # 2) all current RICs observed over time
    if 'RIC_current' in q.columns:
        for val in q['RIC_current']:
            _add('RIC', val)

    # 3) all legacy/raw RICs observed over time
    if 'RIC' in q.columns:
        for val in q['RIC']:
            _add('RIC', val)

    # Include pull_id history too (if present), without duplicating existing keys.
    if 'pull_id' in q.columns and 'id_type' in q.columns:
        for id_type, pull_id in zip(q['id_type'], q['pull_id']):
            it = str(id_type).upper().strip() if pd.notna(id_type) else ''
            if it == 'ISIN':
                norm = _e_norm_isin(pull_id)
                if norm:
                    _add('ISIN', norm)
                    _add('ISIN', f'ISIN:{norm}')
            elif it == 'RIC':
                _add('RIC', pull_id)

    return out


def _e_cache_file(firm_id: str, id_type: str, pull_id: str) -> Path:
    raw = f'{firm_id}|{id_type}|{pull_id}'
    h = hashlib.sha1(raw.encode('utf-8')).hexdigest()[:12]
    clean = re.sub(r'[^A-Za-z0-9._-]', '_', firm_id)
    preferred = CACHE_DIR_C / f"{clean[:70]}__{id_type}_{h}__{CACHE_VERSION_E}.parquet"
    if preferred.exists():
        return preferred

    matches = sorted(CACHE_DIR_C.glob(f"*__{id_type}_{h}__{CACHE_VERSION_E}.parquet"))
    if matches:
        return matches[0]

    legacy = sorted(CACHE_DIR_C.glob(f"*__{id_type}_{h}.parquet"))
    if legacy:
        return legacy[0]

    return preferred


def _e_load_cache(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=['date'] + TARGET_COLS_E)
    d = pd.read_parquet(path).copy()
    d = _e_flatten_cols(d)
    if 'date' in d.columns:
        d['date'] = pd.to_datetime(d['date'], errors='coerce').dt.normalize()
    else:
        d['date'] = pd.NaT

    for c in TARGET_COLS_E:
        if c not in d.columns:
            d[c] = np.nan
        d[c] = pd.to_numeric(d[c], errors='coerce')

    d = d.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return d[['date'] + TARGET_COLS_E]


def _e_save_cache(path: Path, d: pd.DataFrame) -> None:
    tmp = path.with_suffix(path.suffix + '.tmp')
    out = d.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce').dt.normalize()
    for c in TARGET_COLS_E:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors='coerce')
    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    out.to_parquet(tmp, index=False)
    tmp.replace(path)


def _e_cache_covers_range(cached: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> bool:
    if cached is None or cached.empty:
        return False
    cmin = cached['date'].min()
    cmax = cached['date'].max()
    if pd.isna(cmin) or pd.isna(cmax):
        return False

    has_any_cached_value = bool(cached[REQUIRED_COLS_E].notna().any().any())
    if not has_any_cached_value:
        return False

    return bool((cmin <= start) and (cmax >= end))


def _e_combine(parts: list[pd.DataFrame]) -> pd.DataFrame:
    recs = []
    for part in parts:
        if part is None or len(part) == 0:
            continue
        g = _e_flatten_cols(part.copy())
        for c in ['date'] + TARGET_COLS_E:
            if c not in g.columns:
                g[c] = np.nan
        g['date'] = pd.to_datetime(g['date'], errors='coerce').dt.normalize()
        for c in TARGET_COLS_E:
            g[c] = pd.to_numeric(g[c], errors='coerce')
        g = g[['date'] + TARGET_COLS_E]
        recs.extend(g.to_dict('records'))

    if not recs:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_E)

    out = pd.DataFrame.from_records(recs, columns=['date'] + TARGET_COLS_E)
    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return out


def _e_extract_targets_history(raw: pd.DataFrame) -> pd.DataFrame:
    if raw is None or len(raw) == 0:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_E)

    x = _e_flatten_cols(pd.DataFrame(raw).copy().reset_index())
    if x.empty:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_E)

    date_col = None
    for c in x.columns:
        uc = str(c).upper()
        if 'PERIOD' in uc and 'DATE' in uc:
            date_col = c
            break
    if date_col is None:
        for c in x.columns:
            if 'date' in str(c).lower():
                date_col = c
                break
    if date_col is None:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_E)

    out = pd.DataFrame({'date': pd.to_datetime(x[date_col], errors='coerce').dt.normalize()})

    def _norm(txt: str) -> str:
        return re.sub(r'[^A-Z0-9]', '', str(txt).upper())

    def _field_token(field_expr: str) -> str:
        m = re.search(r'TR\.F\.([A-Z0-9_]+)', str(field_expr), flags=re.IGNORECASE)
        return _norm(m.group(1)) if m else _norm(field_expr)

    fallback_tokens = {
        'Dividends': ['DIVPAIDCASHTOTCF', 'DIVIDENDS', 'DIVPAID'],
        'Buybacks': ['COMSTOCKBUYBACKNET', 'BUYBACK', 'REPURCHASE'],
    }

    norm_cols = {c: _norm(c) for c in x.columns}

    for tgt, field_expr in TARGET_FIELDS_E.items():
        token = _field_token(field_expr)
        cand = None
        for c, cn in norm_cols.items():
            if token and (token in cn or cn in token):
                cand = c
                break
        if cand is None:
            for alt in fallback_tokens.get(tgt, []):
                alt_n = _norm(alt)
                for c, cn in norm_cols.items():
                    if alt_n and alt_n in cn:
                        cand = c
                        break
                if cand is not None:
                    break
        out[tgt] = pd.to_numeric(x[cand], errors='coerce') if cand is not None else np.nan

    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return out[['date'] + TARGET_COLS_E]


def _e_get_data_with_retries(universe: list[str], fields: list[str], params: dict | None = None, dbg_label: str = '') -> pd.DataFrame:
    last_err = None
    for r in range(MAX_RETRIES_E + 1):
        try:
            raw = ld.get_data(universe=universe, fields=fields, parameters=(params or {}))
            return pd.DataFrame(raw)
        except Exception as e:
            last_err = e
            if r >= MAX_RETRIES_E:
                break
            time.sleep(BASE_SLEEP_E * (2 ** r) + random.random() * 0.5)
    if last_err is not None:
        msg = str(last_err)
        # For unresolved identifiers we keep logs at company-level only.
        if 'Unable to resolve all requested identifiers' not in msg:
            print(f'[WARN get_data failed] {dbg_label}: {type(last_err).__name__}: {last_err}')
    return pd.DataFrame()


def _e_pull_targets_multi_segment(pull_ids: list[str], start: pd.Timestamp, end: pd.Timestamp) -> dict[str, pd.DataFrame]:
    uniq = []
    seen = set()
    for pid in pull_ids:
        p = str(pid).strip()
        if not p or p in seen:
            continue
        seen.add(p)
        uniq.append(p)

    out = {pid: pd.DataFrame(columns=['date'] + TARGET_COLS_E) for pid in uniq}
    if not uniq:
        return out

    fields = ['TR.F.PeriodEndDate(Period=FY0)'] + list(TARGET_FIELDS_E.values())
    params = {
        'SDate': start.strftime('%Y-%m-%d'),
        'EDate': end.strftime('%Y-%m-%d'),
        'FRQ': 'FY',
    }

    for i in range(0, len(uniq), MULTI_UNIVERSE_CHUNK_E):
        chunk = uniq[i:i + MULTI_UNIVERSE_CHUNK_E]
        raw = _e_get_data_with_retries(universe=chunk, fields=fields, params=params, dbg_label=f'chunk={i//MULTI_UNIVERSE_CHUNK_E+1}')
        if raw is None or raw.empty:
            continue

        z = _e_flatten_cols(raw.copy())
        if z.empty:
            continue

        inst_col = None
        for c in z.columns:
            cl = c.lower()
            if cl == 'instrument' or 'instrument' in cl:
                inst_col = c
                break

        if inst_col is None:
            # If request had a single identifier and provider omitted instrument column,
            # attribute whole frame to that identifier.
            if len(chunk) == 1:
                hist = _e_extract_targets_history(z)
                if not hist.empty:
                    out[chunk[0]] = _e_combine([out[chunk[0]], hist])
            continue

        inst_norm = z[inst_col].astype('string').str.strip().str.upper()

        for pid in chunk:
            pnorm = str(pid).strip().upper()
            g = z.loc[inst_norm == pnorm].copy()
            if g.empty:
                # Some providers return decorated instruments; try contains as fallback.
                g = z.loc[inst_norm.str.contains(re.escape(pnorm), regex=True, na=False)].copy()
            if g.empty:
                # If single-id chunk and instrument mismatch, still use full frame.
                if len(chunk) == 1:
                    g = z.copy()
                else:
                    continue

            hist = _e_extract_targets_history(g)
            if not hist.empty:
                out[pid] = _e_combine([out[pid], hist])

    return out


def _e_update_company_cache(firm_id: str, id_type: str, pull_id: str, start: pd.Timestamp, end: pd.Timestamp, force_refresh: bool = False) -> pd.DataFrame:
    path = _e_cache_file(firm_id, id_type, pull_id)
    cached = pd.DataFrame(columns=['date'] + TARGET_COLS_E) if force_refresh else _e_load_cache(path)

    if _e_cache_covers_range(cached, start, end) and (not force_refresh):
        return cached

    if CACHE_ONLY_E:
        return cached

    pulled = _e_pull_targets_multi_segment([pull_id], start, end).get(pull_id, pd.DataFrame(columns=['date'] + TARGET_COLS_E))
    combined = _e_combine([cached, pulled])
    if not combined.empty or force_refresh:
        _e_save_cache(path, combined)
    return _e_load_cache(path)


def _e_map_to_asof(req_dates: pd.Series, hist: pd.DataFrame, tol_days: int) -> pd.DataFrame:
    left = pd.DataFrame({'date': pd.to_datetime(req_dates, errors='coerce').dt.normalize()}).dropna().sort_values('date')
    if left.empty:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_E)

    if hist is None or hist.empty:
        for c in TARGET_COLS_E:
            left[c] = np.nan
        return left

    right = hist[['date'] + TARGET_COLS_E].copy().sort_values('date').drop_duplicates(['date'], keep='last')
    out = pd.merge_asof(
        left,
        right,
        on='date',
        direction='backward',
        tolerance=pd.Timedelta(days=ASOF_TOL_DAYS_E),
    )
    return out


# -------------------------
# Build base request panel
# -------------------------
base_e = pd.read_parquet(BASE_PATH_E).copy()
base_e['date'] = _e_resolve_asof(base_e)

for c in ['firm_id', 'ISIN', 'RIC_current', 'RIC', 'id_type', 'pull_id', 'firm_id']:
    if c in base_e.columns:
        base_e[c] = _e_clean_str(base_e[c])

if 'id_type' not in base_e.columns:
    base_e['id_type'] = np.select(
        [base_e.get('ISIN', pd.Series(pd.NA, index=base_e.index)).notna(),
         base_e.get('RIC_current', pd.Series(pd.NA, index=base_e.index)).notna(),
         base_e.get('RIC', pd.Series(pd.NA, index=base_e.index)).notna()],
        ['ISIN', 'RIC', 'RIC'],
        default=pd.NA,
    )
if 'pull_id' not in base_e.columns:
    base_e['pull_id'] = np.select(
        [base_e.get('ISIN', pd.Series(pd.NA, index=base_e.index)).notna(),
         base_e.get('RIC_current', pd.Series(pd.NA, index=base_e.index)).notna(),
         base_e.get('RIC', pd.Series(pd.NA, index=base_e.index)).notna()],
        [base_e.get('ISIN', pd.Series(pd.NA, index=base_e.index)),
         base_e.get('RIC_current', pd.Series(pd.NA, index=base_e.index)),
         base_e.get('RIC', pd.Series(pd.NA, index=base_e.index))],
        default=pd.NA,
    )
base_e['id_type'] = _e_clean_str(base_e['id_type'])
base_e['pull_id'] = _e_clean_str(base_e['pull_id'])

if 'firm_id' not in base_e.columns:
    if 'firm_id' in base_e.columns and base_e['firm_id'].notna().any():
        base_e['firm_id'] = _e_clean_str(base_e['firm_id'])
    else:
        ck = _e_clean_str(base_e.get('ISIN', pd.Series(pd.NA, index=base_e.index, dtype='string')))
        ck = ck.fillna(_e_clean_str(base_e.get('RIC_current', pd.Series(pd.NA, index=base_e.index, dtype='string'))))
        ck = ck.fillna(_e_clean_str(base_e.get('RIC', pd.Series(pd.NA, index=base_e.index, dtype='string'))))
        base_e['firm_id'] = 'CID:' + ck.astype('string')

req_cols_e = [c for c in ['firm_id', 'date', 'ISIN', 'RIC_current', 'RIC', 'id_type', 'pull_id'] if c in base_e.columns]
req_e = (
    base_e[req_cols_e]
    .dropna(subset=['firm_id', 'date'])
    .drop_duplicates(['firm_id', 'date'], keep='last')
    .reset_index(drop=True)
)
if req_e.empty:
    raise ValueError('No valid (firm_id, date) rows found for Step 7.')

company_candidates_map = {}
for ck, g in req_e.groupby('firm_id', sort=False):
    company_candidates_map[str(ck)] = _e_build_company_candidates(g)

req_e['n_id_candidates'] = req_e['firm_id'].astype(str).map(lambda ck: len(company_candidates_map.get(ck, [])))

print('Step 7 request rows (company x asof):', len(req_e))
print('As-of range:', req_e['date'].min(), 'to', req_e['date'].max())
print('Unique companies:', req_e['firm_id'].nunique())
print('Active LSEG fields:', TARGET_FIELDS_E)
print('Mode:', 'CACHE_ONLY' if CACHE_ONLY_E else 'CACHE+NETWORK')
print('ID fallback order (historical per firm): all ISINs -> all RIC_current -> all RIC (+ pull_id/id_type history)')


# -------------------------
# Pull loop (batched + resume)
# -------------------------
companies_all_e = req_e['firm_id'].dropna().unique().tolist()
N_total_e = len(companies_all_e)

existing_step_rows_e = pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS_E)
if STEP7_ROWS_PATH.exists():
    try:
        existing_step_rows_e = pd.read_parquet(STEP7_ROWS_PATH)
        if 'date' in existing_step_rows_e.columns:
            existing_step_rows_e['date'] = pd.to_datetime(existing_step_rows_e['date'], errors='coerce').dt.normalize()
        for c in TARGET_COLS_E:
            if c not in existing_step_rows_e.columns:
                existing_step_rows_e[c] = np.nan
            existing_step_rows_e[c] = pd.to_numeric(existing_step_rows_e[c], errors='coerce')
        existing_step_rows_e = existing_step_rows_e[['firm_id', 'date'] + TARGET_COLS_E].dropna(subset=['firm_id', 'date'])
        existing_step_rows_e = existing_step_rows_e.drop_duplicates(['firm_id', 'date'], keep='last')
    except Exception as e:
        print(f'Warning: failed to read STEP7 rows cache, continuing empty: {e}')
        existing_step_rows_e = pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS_E)

processed_from_rows_e = set(existing_step_rows_e['firm_id'].dropna().astype(str).unique().tolist()) if not existing_step_rows_e.empty else set()
processed_from_ckpt_e = set()
if STEP7_CKPT_PATH.exists():
    try:
        ck = json.loads(STEP7_CKPT_PATH.read_text())
        processed_from_ckpt_e = set(str(x) for x in ck.get('processed_companies', []) if str(x).strip())
    except Exception as e:
        print(f'Warning: failed to read checkpoint, ignoring: {e}')

cache_file_count_e = len(list(CACHE_DIR_C.glob('*.parquet')))
if cache_file_count_e == 0 and (processed_from_rows_e or processed_from_ckpt_e):
    print('Step7: cache directory is empty -> ignoring step rows/checkpoint and starting full rebuild.')
    processed_from_rows_e = set()
    processed_from_ckpt_e = set()

processed_companies_e = set(processed_from_rows_e) | set(processed_from_ckpt_e)
companies_e = [c for c in companies_all_e if str(c) not in processed_companies_e]
N_e = len(companies_e)

print('Resume info: total_companies=', N_total_e, 'already_processed=', len(processed_companies_e), 'remaining=', N_e, 'cache_files=', cache_file_count_e)

run_t0_e = time.time()
new_rows_out_e = []
bad_rows_e = []

total_cand_calls_e = 0
total_all2_resolved_e = 0
total_not_all2_e = 0
total_item_found_e = {c: 0 for c in REQUIRED_COLS_E}

if not CACHE_ONLY_E:
    ld.open_session()
try:
    if N_e == 0:
        print('No remaining companies to pull in Step 7.')

    n_batches_e = int(np.ceil(N_e / BATCH_SIZE_E)) if N_e > 0 else 0
    for b_ix, b_start in enumerate(range(0, N_e, BATCH_SIZE_E), start=1):
        b_end = min(N_e, b_start + BATCH_SIZE_E)
        batch_companies = companies_e[b_start:b_end]
        batch_t0 = time.time()
        batch_new_rows = []
        batch_processed = []

        print(f'[BATCH {b_ix}/{n_batches_e}] companies={len(batch_companies)} idx={b_start+1}-{b_end}')

        for k, firm_id in enumerate(batch_companies, start=1):
            q = req_e[req_e['firm_id'] == firm_id].copy().sort_values('date')
            dates = q['date'].dropna().drop_duplicates().sort_values().reset_index(drop=True)
            if dates.empty:
                continue

            start = pd.to_datetime(dates.min()).normalize()
            end = pd.to_datetime(dates.max()).normalize()

            panel = pd.DataFrame({'date': dates})
            for c in TARGET_COLS_E:
                panel[c] = np.nan

            cands = company_candidates_map.get(str(firm_id), [])
            cand_used = 0
            attempted_ids = []
            err_msgs = []

            for cand in cands:
                if panel[REQUIRED_COLS_E].notna().all(axis=1).all():
                    break
                if (not isinstance(cand, (list, tuple))) or len(cand) != 2:
                    continue

                id_type = str(cand[0]).upper().strip()
                pull_id = str(cand[1]).strip()
                if not id_type or not pull_id:
                    continue

                cand_used += 1
                total_cand_calls_e += 1
                attempted_ids.append(pull_id if str(pull_id).upper().startswith('ISIN:') else f'{id_type}:{pull_id}')
                cache_path = _e_cache_file(str(firm_id), id_type, pull_id)
                cached_pre = _e_load_cache(cache_path)
                if _e_cache_covers_range(cached_pre, start, end) and (not FORCE_REFRESH_E):
                    hist = cached_pre
                else:
                    hist = _e_update_company_cache(
                        firm_id=str(firm_id),
                        id_type=id_type,
                        pull_id=pull_id,
                        start=start,
                        end=end,
                        force_refresh=FORCE_REFRESH_E,
                    )
                mapped = _e_map_to_asof(panel['date'], hist, tol_days=ASOF_TOL_DAYS_E)
                if mapped.empty:
                    continue

                for c in TARGET_COLS_E:
                    cur = pd.to_numeric(panel[c], errors='coerce')
                    nxt = pd.to_numeric(mapped.get(c, np.nan), errors='coerce')
                    panel[c] = cur.where(cur.notna(), nxt)

            panel.insert(0, 'firm_id', firm_id)
            recs = panel[['firm_id', 'date'] + TARGET_COLS_E].to_dict('records')
            new_rows_out_e.extend(recs)
            batch_new_rows.extend(recs)
            batch_processed.append(str(firm_id))

            all2_resolved = int(panel[REQUIRED_COLS_E].notna().all(axis=1).sum())
            not_all2 = int(len(panel) - all2_resolved)
            item_found = {c: int(panel[c].notna().sum()) for c in REQUIRED_COLS_E}
            total_all2_resolved_e += all2_resolved
            total_not_all2_e += not_all2
            for c in REQUIRED_COLS_E:
                total_item_found_e[c] += item_found[c]

            any_required_found = bool(panel[REQUIRED_COLS_E].notna().any().any())
            # Warn only if all 3 fallback stages were attempted and none produced data.
            if (not any_required_found) and (cand_used >= 3):
                print(
                    f'[WARN company no data across all 3 fallback identifiers] company_id={str(firm_id)[:40]} '
                    f'ids={attempted_ids[:4]}'
                )

            unresolved = panel[panel[REQUIRED_COLS_E].isna().any(axis=1)]
            if not unresolved.empty:
                bad_rows_e.extend(
                    unresolved[['firm_id', 'date']]
                    .assign(reason='not_all2_resolved', n_candidates=int(len(cands)))
                    .to_dict('records')
                )

            elapsed = time.time() - run_t0_e
            print(
                f'[BATCH {b_ix}/{n_batches_e}] [{b_start+k}/{N_e}] company={str(firm_id)[:40]} rows={len(dates)} '
                f'cand_used={cand_used}/{len(cands)} all2_resolved={all2_resolved} not_all2={not_all2} '
                f'found_Dividends={item_found.get("Dividends",0)} found_Buybacks={item_found.get("Buybacks",0)} '
                f'elapsed={elapsed/60:.1f}m'
            )

        if batch_new_rows:
            batch_df = pd.DataFrame(batch_new_rows)
            batch_df['date'] = pd.to_datetime(batch_df['date'], errors='coerce').dt.normalize()
            for c in TARGET_COLS_E:
                batch_df[c] = pd.to_numeric(batch_df[c], errors='coerce')
            batch_df = batch_df.dropna(subset=['firm_id', 'date'])

            if STEP7_ROWS_PATH.exists():
                prev = pd.read_parquet(STEP7_ROWS_PATH)
                if prev is None or prev.empty:
                    combined = batch_df.copy()
                else:
                    prev = prev.copy()
                    prev['date'] = pd.to_datetime(prev.get('date'), errors='coerce').dt.normalize()
                    for c in TARGET_COLS_E:
                        if c not in prev.columns:
                            prev[c] = np.nan
                        prev[c] = pd.to_numeric(prev[c], errors='coerce')
                    prev = prev[['firm_id', 'date'] + TARGET_COLS_E].dropna(subset=['firm_id', 'date'])
                    combined = pd.concat([prev, batch_df], ignore_index=True)
            else:
                combined = batch_df.copy()

            combined = combined.sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last')
            combined.to_parquet(STEP7_ROWS_PATH, index=False)

        processed_companies_e.update(batch_processed)
        ckpt_payload = {
            'processed_companies': sorted(processed_companies_e),
            'last_batch': b_ix,
            'remaining_companies': max(0, N_total_e - len(processed_companies_e)),
            'updated_at_utc': pd.Timestamp.utcnow().isoformat(),
        }
        STEP7_CKPT_PATH.write_text(json.dumps(ckpt_payload, ensure_ascii=False, indent=2))

        batch_elapsed = (time.time() - batch_t0) / 60
        print(f'[BATCH {b_ix}/{n_batches_e} DONE] processed={len(batch_processed)} batch_elapsed={batch_elapsed:.1f}m')

        if b_end < N_e and BATCH_PAUSE_SEC_E > 0:
            print(f'[BATCH PAUSE] sleeping {BATCH_PAUSE_SEC_E:.0f}s before next batch...')
            time.sleep(BATCH_PAUSE_SEC_E)
finally:
    if not CACHE_ONLY_E:
        ld.close_session()

print(
    f'Done Step 7 (Module C): total_companies={N_total_e}, run_remaining_start={N_e}, candidate_calls={total_cand_calls_e}, '
    f'all2_resolved_rows={total_all2_resolved_e}, not_all2_rows={total_not_all2_e}, '
    f'found_Dividends={total_item_found_e.get("Dividends",0)}, found_Buybacks={total_item_found_e.get("Buybacks",0)}'
)


rebuild_output_e = FORCE_REFRESH_E or (N_e > 0) or (not STEP7_ROWS_PATH.exists()) or (not OUTPUT_PATH_COMBINED.exists())
if (not rebuild_output_e) and BASE_PATH_E.exists() and OUTPUT_PATH_COMBINED.exists():
    rebuild_output_e = BASE_PATH_E.stat().st_mtime > OUTPUT_PATH_COMBINED.stat().st_mtime

if (not rebuild_output_e) and OUTPUT_PATH_COMBINED.exists():
    try:
        chk = pd.read_parquet(OUTPUT_PATH_COMBINED)
        needed_e = {'Dividends', 'Buybacks'}
        if not needed_e.issubset(set(chk.columns)):
            rebuild_output_e = True
            print('Step 7 rebuild forced: combined output missing Module-C columns.')
    except Exception:
        rebuild_output_e = True
        print('Step 7 rebuild forced: unable to inspect combined output file.')

# -------------------------
# Build final output table
# -------------------------
if rebuild_output_e and STEP7_ROWS_PATH.exists():
    out_panel_e = pd.read_parquet(STEP7_ROWS_PATH)
else:
    out_panel_e = pd.DataFrame(new_rows_out_e)

if rebuild_output_e and out_panel_e.empty:
    out_panel_e = pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS_E)
elif rebuild_output_e:
    out_panel_e['date'] = pd.to_datetime(out_panel_e['date'], errors='coerce').dt.normalize()
    for c in TARGET_COLS_E:
        if c not in out_panel_e.columns:
            out_panel_e[c] = np.nan
        out_panel_e[c] = pd.to_numeric(out_panel_e[c], errors='coerce')
    out_panel_e = out_panel_e[['firm_id', 'date'] + TARGET_COLS_E]
    out_panel_e = out_panel_e.sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last')

if rebuild_output_e:
    # Start from base euro500 and preserve existing loaded columns from combined output (if present).
    out_e = base_e.copy()
    if OUTPUT_PATH_COMBINED.exists():
        old = pd.read_parquet(OUTPUT_PATH_COMBINED).copy()
        old['date'] = _e_resolve_asof(old)

        keep_old = [c for c in old.columns if c not in out_e.columns and c not in {'firm_id', 'date', 'firm_id'}]
        if keep_old:
            # Stage 1: preserve on firm_id + date
            if 'firm_id' in out_e.columns and 'firm_id' in old.columns:
                old_firm = old[['firm_id', 'date'] + keep_old].dropna(subset=['firm_id', 'date'])
                old_firm = old_firm.drop_duplicates(['firm_id', 'date'], keep='last')
                out_e = out_e.merge(old_firm, on=['firm_id', 'date'], how='left', suffixes=('', '_oldfirm'))

            # Stage 2 fallback: preserve on firm_id + date
            old_small = old[['firm_id', 'date'] + keep_old].dropna(subset=['firm_id', 'date'])
            old_small = old_small.drop_duplicates(['firm_id', 'date'], keep='last')
            add_old = base_e.merge(old_small, on=['firm_id', 'date'], how='left', suffixes=('', '_oldkey'))

            for c in keep_old:
                old_s = out_e[c] if c in out_e.columns else pd.Series(pd.NA, index=out_e.index, dtype='object')
                old_firm_s = out_e.get(f'{c}_oldfirm', pd.Series(pd.NA, index=out_e.index, dtype='object'))
                old_key_s = add_old.get(f'{c}_oldkey', pd.Series(pd.NA, index=out_e.index, dtype='object'))
                out_e[c] = old_firm_s.fillna(old_key_s).fillna(old_s)
                if f'{c}_oldfirm' in out_e.columns:
                    out_e = out_e.drop(columns=[f'{c}_oldfirm'])

    # Stage 1: preferred merge on firm_id + date
    if 'firm_id' in base_e.columns:
        out_panel_e_firm = out_panel_e.rename(columns={'firm_id': 'firm_id'})
        out_e = out_e.merge(out_panel_e_firm, on=['firm_id', 'date'], how='left', suffixes=('', '_firm'))

    # Stage 2 fallback: merge on firm_id + date
    add_e = base_e.merge(out_panel_e, on=['firm_id', 'date'], how='left', suffixes=('', '_key'))

    for c in TARGET_COLS_E:
        old_s = pd.to_numeric(out_e[c], errors='coerce') if c in out_e.columns else pd.Series(np.nan, index=out_e.index)
        new_firm_s = pd.to_numeric(
            out_e[f'{c}_firm'] if f'{c}_firm' in out_e.columns else pd.Series(np.nan, index=out_e.index),
            errors='coerce'
        )
        new_key_s = pd.to_numeric(
            add_e[f'{c}_key'] if f'{c}_key' in add_e.columns else pd.Series(np.nan, index=out_e.index),
            errors='coerce'
        )
        out_e[c] = new_firm_s.combine_first(new_key_s).combine_first(old_s)
        if f'{c}_firm' in out_e.columns:
            out_e = out_e.drop(columns=[f'{c}_firm'])

    # Keep only canonical beta column in combined output
    if ('beta' not in out_e.columns) and ('beta_3m' in out_e.columns):
        out_e['beta'] = pd.to_numeric(out_e['beta_3m'], errors='coerce')
    if 'beta_3m' in out_e.columns:
        out_e = out_e.drop(columns=['beta_3m'])

    out_e.to_parquet(OUTPUT_PATH_COMBINED, index=False)
    euro500_netpayout_df = out_e.copy()

    print('Saved combined output (Step 5 + 6 + 7 / Module C):', OUTPUT_PATH_COMBINED, 'rows:', len(out_e))
    print('Data Wrangler variable ready: euro500_netpayout_df')
else:
    out_e = pd.read_parquet(OUTPUT_PATH_COMBINED)
    euro500_netpayout_df = out_e.copy()
    print('Skipped Step-7 rebuild (already up-to-date):', OUTPUT_PATH_COMBINED, 'rows:', len(out_e))
    print('Data Wrangler variable ready: euro500_netpayout_df')

if bad_rows_e:
    bad_df = pd.DataFrame(bad_rows_e)
    if BAD_LOG_PATH_C.exists():
        old = pd.read_csv(BAD_LOG_PATH_C)
        for c in ['firm_id', 'date', 'reason', 'n_candidates']:
            if c not in old.columns:
                old[c] = pd.NA
            if c not in bad_df.columns:
                bad_df[c] = pd.NA
        out_bad = pd.DataFrame.from_records(
            old[['firm_id', 'date', 'reason', 'n_candidates']].to_dict('records')
            + bad_df[['firm_id', 'date', 'reason', 'n_candidates']].to_dict('records'),
            columns=['firm_id', 'date', 'reason', 'n_candidates'],
        )
    else:
        out_bad = bad_df

    out_bad['date'] = pd.to_datetime(out_bad['date'], errors='coerce').dt.normalize()
    out_bad = out_bad.drop_duplicates(subset=['firm_id', 'date', 'reason'], keep='last')
    out_bad.to_csv(BAD_LOG_PATH_C, index=False)
    print('Updated Module-C bad-id log:', BAD_LOG_PATH_C, 'rows:', len(out_bad))







### 4B. Coverage-Analyse Modul C (`Dividends`, `Buybacks`)

In [ ]:
# Robust path resolution for Step 7B
MODULEC_OUTPUT_PATH = globals().get('OUTPUT_PATH_COMBINED_C', DATA_DIR / 'euro500_netpayout.parquet')

if not MODULEC_OUTPUT_PATH.exists():
    raise FileNotFoundError(f'Missing file: {MODULEC_OUTPUT_PATH}')

mc = pd.read_parquet(MODULEC_OUTPUT_PATH).copy()
if 'date' in mc.columns:
    mc['date'] = pd.to_datetime(mc['date'], errors='coerce')

targets_c = [c for c in ['Dividends', 'Buybacks'] if c in mc.columns]
if not targets_c:
    raise KeyError('No Module C columns found (expected Dividends/Buybacks). Run Step 7 first.')

for c in targets_c:
    mc[c] = pd.to_numeric(mc[c], errors='coerce')

if 'date' in mc.columns and mc['date'].notna().any():
    mc['year'] = mc['date'].dt.year
else:
    mc['year'] = pd.NA

mc['all2_available'] = mc[targets_c].notna().all(axis=1)

overall_c = {'rows_total': len(mc)}
for c in targets_c:
    overall_c[f'cov_{c}'] = float(mc[c].notna().mean()) if len(mc) else np.nan
overall_c['cov_all2'] = float(mc['all2_available'].mean()) if len(mc) else np.nan

display(pd.DataFrame([overall_c]))

if mc['year'].notna().any():
    yc = mc.groupby('year', as_index=False).agg(rows=('year', 'size'))
    for c in targets_c:
        yc[f'cov_{c}'] = mc.groupby('year')[c].apply(lambda s: s.notna().mean()).values
    yc['cov_all2'] = mc.groupby('year')['all2_available'].apply(lambda s: float(s.mean())).values

    plt.figure(figsize=(11, 5))
    for c in targets_c:
        plt.plot(yc['year'], yc[f'cov_{c}'], marker='o', linewidth=1.6, label=c)
    plt.title('Module C yearly coverage by item')
    plt.xlabel('Year')
    plt.ylabel('Coverage share')
    plt.ylim(0, 1.02)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()


    # Zusatzplot: Module C coverage (Non-Financials only)
    if 'trbc_sector' in mc.columns:
        mc_nf = mc[mc['trbc_sector'].astype(str).str.strip().str.lower() != 'financials'].copy()
        if mc_nf['year'].notna().any() and not mc_nf.empty:
            yc_nf = mc_nf.groupby('year', as_index=False).agg(rows=('year', 'size'))
            for c in targets_c:
                yc_nf[f'cov_{c}'] = mc_nf.groupby('year')[c].apply(lambda s: s.notna().mean()).values
            yc_nf['cov_all2'] = mc_nf.groupby('year')['all2_available'].apply(lambda s: float(s.mean())).values

            plt.figure(figsize=(11, 5))
            for c in targets_c:
                plt.plot(yc_nf['year'], yc_nf[f'cov_{c}'], marker='o', linewidth=1.6, label=c)
            plt.title('Module C yearly coverage by item (Non-Financials)')
            plt.xlabel('Year')
            plt.ylabel('Coverage share')
            plt.ylim(0, 1.02)
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.show()
        else:
            print('No valid Non-Financials rows/year information for Module C coverage plot.')
    else:
        print('Column trbc_sector missing; skipping Module C Non-Financials coverage plot.')
else:
    print('No valid year information available for yearly coverage plot.')





## 5. Modul D: Cash & ShortTermInvestments (FY)

In [ ]:
# ------------------------------------------------------------
# Step 7D — Module D: Cash & ShortTermInvestments (FY): CashSTInvst
#   Input  : euro500.parquet
#   Output : euro500_netpayout.parquet (combined table)
# ------------------------------------------------------------

# -------------------------
# Config
# -------------------------
BASE_PATH_D = DATA_DIR / 'euro500.parquet'
OUTPUT_PATH_COMBINED_D = DATA_DIR / 'euro500_netpayout.parquet'

CACHE_DIR_D = CACHE_DATA_DIR / 'moduleD_cache_by_company_id'
CACHE_DIR_D.mkdir(parents=True, exist_ok=True)
BAD_LOG_PATH_D = CACHE_DATA_DIR / 'moduleD_bad_ids.csv'
STEP7D_ROWS_PATH = CACHE_DATA_DIR / 'moduleD_step7d_rows.parquet'
STEP7D_CKPT_PATH = CACHE_DATA_DIR / 'moduleD_step7d_checkpoint.json'

TARGET_FIELDS_D = {
    'CashSTInvst': 'TR.F.CashSTInvst(Period=FY0)',
}
TARGET_COLS_D = list(TARGET_FIELDS_D.keys())
REQUIRED_COLS_D = ['CashSTInvst']

ASOF_TOL_DAYS_D = 365
MAX_RETRIES_D = 2
BASE_SLEEP_D = 1.0
FORCE_REFRESH_D = False
CACHE_ONLY_D = False  # False => allow LSEG pull (cache still used)
CACHE_VERSION_D = 'v1'

BATCH_SIZE_D = 100
BATCH_PAUSE_SEC_D = 0
MULTI_UNIVERSE_CHUNK_D = 25


if not BASE_PATH_D.exists():
    raise FileNotFoundError(f'Missing file: {BASE_PATH_D}')


# -------------------------
# Helpers
# -------------------------
def _d_clean_str(s: pd.Series) -> pd.Series:
    x = s.astype('string').str.strip()
    return x.where(x.notna() & (x != ''), pd.NA)


def _d_resolve_asof(df: pd.DataFrame) -> pd.Series:
    if 'date' not in df.columns:
        raise ValueError('Missing required date column.')
    return pd.to_datetime(df['date'], errors='coerce').dt.normalize()


def _d_flatten_cols(df: pd.DataFrame) -> pd.DataFrame:
    x = df.copy()
    if isinstance(x.columns, pd.MultiIndex):
        x.columns = [' | '.join([str(v) for v in tup if v is not None]).strip() for tup in x.columns]
    else:
        x.columns = [str(c).strip() for c in x.columns]
    return x


def _d_norm_isin(value) -> str | None:
    if pd.isna(value):
        return None
    v = str(value).strip()
    if not v:
        return None
    return v.split(':', 1)[1].strip() if v.upper().startswith('ISIN:') else v


def _d_build_company_candidates(company_req: pd.DataFrame) -> list[tuple[str, str]]:
    """Collect all unique identifiers for one company across time.

    Order: all ISINs, then all RIC_current, then all RIC (chronological first-seen).
    """
    q = company_req.copy().sort_values('date')

    out = []
    seen = set()

    def _add(id_type: str, value):
        if pd.isna(value):
            return
        v = str(value).strip()
        if not v:
            return
        key = (id_type, v)
        if key in seen:
            return
        seen.add(key)
        out.append(key)

    # 1) all ISINs observed over time for this company
    if 'ISIN' in q.columns:
        for val in q['ISIN']:
            norm = _d_norm_isin(val)
            if norm:
                _add('ISIN', norm)
                _add('ISIN', f'ISIN:{norm}')

    # 2) all current RICs observed over time
    if 'RIC_current' in q.columns:
        for val in q['RIC_current']:
            _add('RIC', val)

    # 3) all legacy/raw RICs observed over time
    if 'RIC' in q.columns:
        for val in q['RIC']:
            _add('RIC', val)

    # Include pull_id history too (if present), without duplicating existing keys.
    if 'pull_id' in q.columns and 'id_type' in q.columns:
        for id_type, pull_id in zip(q['id_type'], q['pull_id']):
            it = str(id_type).upper().strip() if pd.notna(id_type) else ''
            if it == 'ISIN':
                norm = _d_norm_isin(pull_id)
                if norm:
                    _add('ISIN', norm)
                    _add('ISIN', f'ISIN:{norm}')
            elif it == 'RIC':
                _add('RIC', pull_id)

    return out


def _d_cache_file(firm_id: str, id_type: str, pull_id: str) -> Path:
    raw = f'{firm_id}|{id_type}|{pull_id}'
    h = hashlib.sha1(raw.encode('utf-8')).hexdigest()[:12]
    clean = re.sub(r'[^A-Za-z0-9._-]', '_', firm_id)
    preferred = CACHE_DIR_D / f"{clean[:70]}__{id_type}_{h}__{CACHE_VERSION_D}.parquet"
    if preferred.exists():
        return preferred

    matches = sorted(CACHE_DIR_D.glob(f"*__{id_type}_{h}__{CACHE_VERSION_D}.parquet"))
    if matches:
        return matches[0]

    legacy = sorted(CACHE_DIR_D.glob(f"*__{id_type}_{h}.parquet"))
    if legacy:
        return legacy[0]

    return preferred


def _d_load_cache(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame(columns=['date'] + TARGET_COLS_D)
    d = pd.read_parquet(path).copy()
    d = _d_flatten_cols(d)
    if 'date' in d.columns:
        d['date'] = pd.to_datetime(d['date'], errors='coerce').dt.normalize()
    else:
        d['date'] = pd.NaT

    for c in TARGET_COLS_D:
        if c not in d.columns:
            d[c] = np.nan
        d[c] = pd.to_numeric(d[c], errors='coerce')

    d = d.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return d[['date'] + TARGET_COLS_D]


def _d_save_cache(path: Path, d: pd.DataFrame) -> None:
    tmp = path.with_suffix(path.suffix + '.tmp')
    out = d.copy()
    out['date'] = pd.to_datetime(out['date'], errors='coerce').dt.normalize()
    for c in TARGET_COLS_D:
        if c not in out.columns:
            out[c] = np.nan
        out[c] = pd.to_numeric(out[c], errors='coerce')
    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    out.to_parquet(tmp, index=False)
    tmp.replace(path)


def _d_cache_covers_range(cached: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp) -> bool:
    if cached is None or cached.empty:
        return False
    cmin = cached['date'].min()
    cmax = cached['date'].max()
    if pd.isna(cmin) or pd.isna(cmax):
        return False

    has_any_cached_value = bool(cached[REQUIRED_COLS_D].notna().any().any())
    if not has_any_cached_value:
        return False

    return bool((cmin <= start) and (cmax >= end))


def _d_combine(parts: list[pd.DataFrame]) -> pd.DataFrame:
    recs = []
    for part in parts:
        if part is None or len(part) == 0:
            continue
        g = _d_flatten_cols(part.copy())
        for c in ['date'] + TARGET_COLS_D:
            if c not in g.columns:
                g[c] = np.nan
        g['date'] = pd.to_datetime(g['date'], errors='coerce').dt.normalize()
        for c in TARGET_COLS_D:
            g[c] = pd.to_numeric(g[c], errors='coerce')
        g = g[['date'] + TARGET_COLS_D]
        recs.extend(g.to_dict('records'))

    if not recs:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_D)

    out = pd.DataFrame.from_records(recs, columns=['date'] + TARGET_COLS_D)
    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return out


def _d_extract_targets_history(raw: pd.DataFrame) -> pd.DataFrame:
    if raw is None or len(raw) == 0:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_D)

    x = _d_flatten_cols(pd.DataFrame(raw).copy().reset_index())
    if x.empty:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_D)

    date_col = None
    for c in x.columns:
        uc = str(c).upper()
        if 'PERIOD' in uc and 'DATE' in uc:
            date_col = c
            break
    if date_col is None:
        for c in x.columns:
            if 'date' in str(c).lower():
                date_col = c
                break
    if date_col is None:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_D)

    out = pd.DataFrame({'date': pd.to_datetime(x[date_col], errors='coerce').dt.normalize()})

    def _norm(txt: str) -> str:
        return re.sub(r'[^A-Z0-9]', '', str(txt).upper())

    def _field_token(field_expr: str) -> str:
        m = re.search(r'TR\.F\.([A-Z0-9_]+)', str(field_expr), flags=re.IGNORECASE)
        return _norm(m.group(1)) if m else _norm(field_expr)

    fallback_tokens = {
        'CashSTInvst': ['CASHSTINVST', 'CASH', 'SHORTTERMINVESTMENTS'],
    }

    norm_cols = {c: _norm(c) for c in x.columns}

    for tgt, field_expr in TARGET_FIELDS_D.items():
        token = _field_token(field_expr)
        cand = None
        for c, cn in norm_cols.items():
            if token and (token in cn or cn in token):
                cand = c
                break
        if cand is None:
            for alt in fallback_tokens.get(tgt, []):
                alt_n = _norm(alt)
                for c, cn in norm_cols.items():
                    if alt_n and alt_n in cn:
                        cand = c
                        break
                if cand is not None:
                    break
        out[tgt] = pd.to_numeric(x[cand], errors='coerce') if cand is not None else np.nan

    out = out.dropna(subset=['date']).sort_values('date').drop_duplicates(['date'], keep='last')
    return out[['date'] + TARGET_COLS_D]


def _d_get_data_with_retries(universe: list[str], fields: list[str], params: dict | None = None, dbg_label: str = '') -> pd.DataFrame:
    last_err = None
    for r in range(MAX_RETRIES_D + 1):
        try:
            raw = ld.get_data(universe=universe, fields=fields, parameters=(params or {}))
            return pd.DataFrame(raw)
        except Exception as e:
            last_err = e
            if r >= MAX_RETRIES_D:
                break
            time.sleep(BASE_SLEEP_D * (2 ** r) + random.random() * 0.5)
    if last_err is not None:
        msg = str(last_err)
        # For unresolved identifiers we keep logs at company-level only.
        if 'Unable to resolve all requested identifiers' not in msg:
            print(f'[WARN get_data failed] {dbg_label}: {type(last_err).__name__}: {last_err}')
    return pd.DataFrame()


def _d_pull_targets_multi_segment(pull_ids: list[str], start: pd.Timestamp, end: pd.Timestamp) -> dict[str, pd.DataFrame]:
    uniq = []
    seen = set()
    for pid in pull_ids:
        p = str(pid).strip()
        if not p or p in seen:
            continue
        seen.add(p)
        uniq.append(p)

    out = {pid: pd.DataFrame(columns=['date'] + TARGET_COLS_D) for pid in uniq}
    if not uniq:
        return out

    fields = ['TR.F.PeriodEndDate(Period=FY0)'] + list(TARGET_FIELDS_D.values())
    params = {
        'SDate': start.strftime('%Y-%m-%d'),
        'EDate': end.strftime('%Y-%m-%d'),
        'FRQ': 'FY',
    }

    for i in range(0, len(uniq), MULTI_UNIVERSE_CHUNK_D):
        chunk = uniq[i:i + MULTI_UNIVERSE_CHUNK_D]
        raw = _d_get_data_with_retries(universe=chunk, fields=fields, params=params, dbg_label=f'chunk={i//MULTI_UNIVERSE_CHUNK_D+1}')
        if raw is None or raw.empty:
            continue

        z = _d_flatten_cols(raw.copy())
        if z.empty:
            continue

        inst_col = None
        for c in z.columns:
            cl = c.lower()
            if cl == 'instrument' or 'instrument' in cl:
                inst_col = c
                break

        if inst_col is None:
            # If request had a single identifier and provider omitted instrument column,
            # attribute whole frame to that identifier.
            if len(chunk) == 1:
                hist = _d_extract_targets_history(z)
                if not hist.empty:
                    out[chunk[0]] = _d_combine([out[chunk[0]], hist])
            continue

        inst_norm = z[inst_col].astype('string').str.strip().str.upper()

        for pid in chunk:
            pnorm = str(pid).strip().upper()
            g = z.loc[inst_norm == pnorm].copy()
            if g.empty:
                # Some providers return decorated instruments; try contains as fallback.
                g = z.loc[inst_norm.str.contains(re.escape(pnorm), regex=True, na=False)].copy()
            if g.empty:
                # If single-id chunk and instrument mismatch, still use full frame.
                if len(chunk) == 1:
                    g = z.copy()
                else:
                    continue

            hist = _d_extract_targets_history(g)
            if not hist.empty:
                out[pid] = _d_combine([out[pid], hist])

    return out


def _d_update_company_cache(firm_id: str, id_type: str, pull_id: str, start: pd.Timestamp, end: pd.Timestamp, force_refresh: bool = False) -> pd.DataFrame:
    path = _d_cache_file(firm_id, id_type, pull_id)
    cached = pd.DataFrame(columns=['date'] + TARGET_COLS_D) if force_refresh else _d_load_cache(path)

    if _d_cache_covers_range(cached, start, end) and (not force_refresh):
        return cached

    if CACHE_ONLY_D:
        return cached

    pulled = _d_pull_targets_multi_segment([pull_id], start, end).get(pull_id, pd.DataFrame(columns=['date'] + TARGET_COLS_D))
    combined = _d_combine([cached, pulled])
    if not combined.empty or force_refresh:
        _d_save_cache(path, combined)
    return _d_load_cache(path)


def _d_map_to_asof(req_dates: pd.Series, hist: pd.DataFrame, tol_days: int) -> pd.DataFrame:
    left = pd.DataFrame({'date': pd.to_datetime(req_dates, errors='coerce').dt.normalize()}).dropna().sort_values('date')
    if left.empty:
        return pd.DataFrame(columns=['date'] + TARGET_COLS_D)

    if hist is None or hist.empty:
        for c in TARGET_COLS_D:
            left[c] = np.nan
        return left

    right = hist[['date'] + TARGET_COLS_D].copy().sort_values('date').drop_duplicates(['date'], keep='last')
    out = pd.merge_asof(
        left,
        right,
        on='date',
        direction='backward',
        tolerance=pd.Timedelta(days=ASOF_TOL_DAYS_D),
    )
    return out


# -------------------------
# Build base request panel
# -------------------------
base_e = pd.read_parquet(BASE_PATH_D).copy()
base_e['date'] = _d_resolve_asof(base_e)

for c in ['firm_id', 'ISIN', 'RIC_current', 'RIC', 'id_type', 'pull_id', 'firm_id']:
    if c in base_e.columns:
        base_e[c] = _d_clean_str(base_e[c])

if 'id_type' not in base_e.columns:
    base_e['id_type'] = np.select(
        [base_e.get('ISIN', pd.Series(pd.NA, index=base_e.index)).notna(),
         base_e.get('RIC_current', pd.Series(pd.NA, index=base_e.index)).notna(),
         base_e.get('RIC', pd.Series(pd.NA, index=base_e.index)).notna()],
        ['ISIN', 'RIC', 'RIC'],
        default=pd.NA,
    )
if 'pull_id' not in base_e.columns:
    base_e['pull_id'] = np.select(
        [base_e.get('ISIN', pd.Series(pd.NA, index=base_e.index)).notna(),
         base_e.get('RIC_current', pd.Series(pd.NA, index=base_e.index)).notna(),
         base_e.get('RIC', pd.Series(pd.NA, index=base_e.index)).notna()],
        [base_e.get('ISIN', pd.Series(pd.NA, index=base_e.index)),
         base_e.get('RIC_current', pd.Series(pd.NA, index=base_e.index)),
         base_e.get('RIC', pd.Series(pd.NA, index=base_e.index))],
        default=pd.NA,
    )
base_e['id_type'] = _d_clean_str(base_e['id_type'])
base_e['pull_id'] = _d_clean_str(base_e['pull_id'])

if 'firm_id' not in base_e.columns:
    if 'firm_id' in base_e.columns and base_e['firm_id'].notna().any():
        base_e['firm_id'] = _d_clean_str(base_e['firm_id'])
    else:
        ck = _d_clean_str(base_e.get('ISIN', pd.Series(pd.NA, index=base_e.index, dtype='string')))
        ck = ck.fillna(_d_clean_str(base_e.get('RIC_current', pd.Series(pd.NA, index=base_e.index, dtype='string'))))
        ck = ck.fillna(_d_clean_str(base_e.get('RIC', pd.Series(pd.NA, index=base_e.index, dtype='string'))))
        base_e['firm_id'] = 'CID:' + ck.astype('string')

req_cols_e = [c for c in ['firm_id', 'date', 'ISIN', 'RIC_current', 'RIC', 'id_type', 'pull_id'] if c in base_e.columns]
req_d = (
    base_e[req_cols_e]
    .dropna(subset=['firm_id', 'date'])
    .drop_duplicates(['firm_id', 'date'], keep='last')
    .reset_index(drop=True)
)
if req_d.empty:
    raise ValueError('No valid (firm_id, date) rows found for Step 7.')

company_candidates_map = {}
for ck, g in req_d.groupby('firm_id', sort=False):
    company_candidates_map[str(ck)] = _d_build_company_candidates(g)

req_d['n_id_candidates'] = req_d['firm_id'].astype(str).map(lambda ck: len(company_candidates_map.get(ck, [])))

print('Step 7 request rows (company x asof):', len(req_d))
print('As-of range:', req_d['date'].min(), 'to', req_d['date'].max())
print('Unique companies:', req_d['firm_id'].nunique())
print('Active LSEG fields:', TARGET_FIELDS_D)
print('Mode:', 'CACHE_ONLY' if CACHE_ONLY_D else 'CACHE+NETWORK')
print('ID fallback order (historical per firm): all ISINs -> all RIC_current -> all RIC (+ pull_id/id_type history)')


# -------------------------
# Pull loop (batched + resume)
# -------------------------
companies_all_e = req_d['firm_id'].dropna().unique().tolist()
N_total_e = len(companies_all_e)

existing_step_rows_d = pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS_D)
if STEP7D_ROWS_PATH.exists():
    try:
        existing_step_rows_d = pd.read_parquet(STEP7D_ROWS_PATH)
        if 'date' in existing_step_rows_d.columns:
            existing_step_rows_d['date'] = pd.to_datetime(existing_step_rows_d['date'], errors='coerce').dt.normalize()
        for c in TARGET_COLS_D:
            if c not in existing_step_rows_d.columns:
                existing_step_rows_d[c] = np.nan
            existing_step_rows_d[c] = pd.to_numeric(existing_step_rows_d[c], errors='coerce')
        existing_step_rows_d = existing_step_rows_d[['firm_id', 'date'] + TARGET_COLS_D].dropna(subset=['firm_id', 'date'])
        existing_step_rows_d = existing_step_rows_d.drop_duplicates(['firm_id', 'date'], keep='last')
    except Exception as e:
        print(f'Warning: failed to read STEP7 rows cache, continuing empty: {e}')
        existing_step_rows_d = pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS_D)

processed_from_rows_e = set(existing_step_rows_d['firm_id'].dropna().astype(str).unique().tolist()) if not existing_step_rows_d.empty else set()
processed_from_ckpt_e = set()
if STEP7D_CKPT_PATH.exists():
    try:
        ck = json.loads(STEP7D_CKPT_PATH.read_text())
        processed_from_ckpt_e = set(str(x) for x in ck.get('processed_companies', []) if str(x).strip())
    except Exception as e:
        print(f'Warning: failed to read checkpoint, ignoring: {e}')

cache_file_count_e = len(list(CACHE_DIR_D.glob('*.parquet')))
if cache_file_count_e == 0 and (processed_from_rows_e or processed_from_ckpt_e):
    print('Step7: cache directory is empty -> ignoring step rows/checkpoint and starting full rebuild.')
    processed_from_rows_e = set()
    processed_from_ckpt_e = set()

processed_companies_e = set(processed_from_rows_e) | set(processed_from_ckpt_e)
companies_e = [c for c in companies_all_e if str(c) not in processed_companies_e]
N_e = len(companies_e)

print('Resume info: total_companies=', N_total_e, 'already_processed=', len(processed_companies_e), 'remaining=', N_e, 'cache_files=', cache_file_count_e)

run_t0_e = time.time()
new_rows_out_d = []
bad_rows_d = []

total_cand_calls_d = 0
total_all_resolved_d = 0
total_not_all_d = 0
total_item_found_d = {c: 0 for c in REQUIRED_COLS_D}

if not CACHE_ONLY_D:
    ld.open_session()
try:
    if N_e == 0:
        print('No remaining companies to pull in Step 7.')

    n_batches_e = int(np.ceil(N_e / BATCH_SIZE_D)) if N_e > 0 else 0
    for b_ix, b_start in enumerate(range(0, N_e, BATCH_SIZE_D), start=1):
        b_end = min(N_e, b_start + BATCH_SIZE_D)
        batch_companies = companies_e[b_start:b_end]
        batch_t0 = time.time()
        batch_new_rows = []
        batch_processed = []

        print(f'[BATCH {b_ix}/{n_batches_e}] companies={len(batch_companies)} idx={b_start+1}-{b_end}')

        for k, firm_id in enumerate(batch_companies, start=1):
            q = req_d[req_d['firm_id'] == firm_id].copy().sort_values('date')
            dates = q['date'].dropna().drop_duplicates().sort_values().reset_index(drop=True)
            if dates.empty:
                continue

            start = pd.to_datetime(dates.min()).normalize()
            end = pd.to_datetime(dates.max()).normalize()

            panel = pd.DataFrame({'date': dates})
            for c in TARGET_COLS_D:
                panel[c] = np.nan

            cands = company_candidates_map.get(str(firm_id), [])
            cand_used = 0
            attempted_ids = []
            err_msgs = []

            for cand in cands:
                if panel[REQUIRED_COLS_D].notna().all(axis=1).all():
                    break
                if (not isinstance(cand, (list, tuple))) or len(cand) != 2:
                    continue

                id_type = str(cand[0]).upper().strip()
                pull_id = str(cand[1]).strip()
                if not id_type or not pull_id:
                    continue

                cand_used += 1
                total_cand_calls_d += 1
                attempted_ids.append(pull_id if str(pull_id).upper().startswith('ISIN:') else f'{id_type}:{pull_id}')
                cache_path = _d_cache_file(str(firm_id), id_type, pull_id)
                cached_pre = _d_load_cache(cache_path)
                if _d_cache_covers_range(cached_pre, start, end) and (not FORCE_REFRESH_D):
                    hist = cached_pre
                else:
                    hist = _d_update_company_cache(
                        firm_id=str(firm_id),
                        id_type=id_type,
                        pull_id=pull_id,
                        start=start,
                        end=end,
                        force_refresh=FORCE_REFRESH_D,
                    )
                mapped = _d_map_to_asof(panel['date'], hist, tol_days=ASOF_TOL_DAYS_D)
                if mapped.empty:
                    continue

                for c in TARGET_COLS_D:
                    cur = pd.to_numeric(panel[c], errors='coerce')
                    nxt = pd.to_numeric(mapped.get(c, np.nan), errors='coerce')
                    panel[c] = cur.where(cur.notna(), nxt)

            panel.insert(0, 'firm_id', firm_id)
            recs = panel[['firm_id', 'date'] + TARGET_COLS_D].to_dict('records')
            new_rows_out_d.extend(recs)
            batch_new_rows.extend(recs)
            batch_processed.append(str(firm_id))

            all_resolved = int(panel[REQUIRED_COLS_D].notna().all(axis=1).sum())
            not_all = int(len(panel) - all_resolved)
            item_found = {c: int(panel[c].notna().sum()) for c in REQUIRED_COLS_D}
            total_all_resolved_d += all_resolved
            total_not_all_d += not_all
            for c in REQUIRED_COLS_D:
                total_item_found_d[c] += item_found[c]

            any_required_found = bool(panel[REQUIRED_COLS_D].notna().any().any())
            # Warn only if all 3 fallback stages were attempted and none produced data.
            if (not any_required_found) and (cand_used >= 3):
                print(
                    f'[WARN company no data across all 3 fallback identifiers] company_id={str(firm_id)[:40]} '
                    f'ids={attempted_ids[:4]}'
                )

            unresolved = panel[panel[REQUIRED_COLS_D].isna().any(axis=1)]
            if not unresolved.empty:
                bad_rows_d.extend(
                    unresolved[['firm_id', 'date']]
                    .assign(reason='not_all_resolved', n_candidates=int(len(cands)))
                    .to_dict('records')
                )

            elapsed = time.time() - run_t0_e
            print(
                f'[BATCH {b_ix}/{n_batches_e}] [{b_start+k}/{N_e}] company={str(firm_id)[:40]} rows={len(dates)} '
                f'cand_used={cand_used}/{len(cands)} all_resolved={all_resolved} not_all={not_all} '
                f'found_CashSTInvst={item_found.get("CashSTInvst",0)} '
                f'elapsed={elapsed/60:.1f}m'
            )

        if batch_new_rows:
            batch_df = pd.DataFrame(batch_new_rows)
            batch_df['date'] = pd.to_datetime(batch_df['date'], errors='coerce').dt.normalize()
            for c in TARGET_COLS_D:
                batch_df[c] = pd.to_numeric(batch_df[c], errors='coerce')
            batch_df = batch_df.dropna(subset=['firm_id', 'date'])

            if STEP7D_ROWS_PATH.exists():
                prev = pd.read_parquet(STEP7D_ROWS_PATH)
                if prev is None or prev.empty:
                    combined = batch_df.copy()
                else:
                    prev = prev.copy()
                    prev['date'] = pd.to_datetime(prev.get('date'), errors='coerce').dt.normalize()
                    for c in TARGET_COLS_D:
                        if c not in prev.columns:
                            prev[c] = np.nan
                        prev[c] = pd.to_numeric(prev[c], errors='coerce')
                    prev = prev[['firm_id', 'date'] + TARGET_COLS_D].dropna(subset=['firm_id', 'date'])
                    combined = pd.concat([prev, batch_df], ignore_index=True)
            else:
                combined = batch_df.copy()

            combined = combined.sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last')
            combined.to_parquet(STEP7D_ROWS_PATH, index=False)

        processed_companies_e.update(batch_processed)
        ckpt_payload = {
            'processed_companies': sorted(processed_companies_e),
            'last_batch': b_ix,
            'remaining_companies': max(0, N_total_e - len(processed_companies_e)),
            'updated_at_utc': pd.Timestamp.utcnow().isoformat(),
        }
        STEP7D_CKPT_PATH.write_text(json.dumps(ckpt_payload, ensure_ascii=False, indent=2))

        batch_dlapsed = (time.time() - batch_t0) / 60
        print(f'[BATCH {b_ix}/{n_batches_e} DONE] processed={len(batch_processed)} batch_dlapsed={batch_dlapsed:.1f}m')

        if b_end < N_e and BATCH_PAUSE_SEC_D > 0:
            print(f'[BATCH PAUSE] sleeping {BATCH_PAUSE_SEC_D:.0f}s before next batch...')
            time.sleep(BATCH_PAUSE_SEC_D)
finally:
    if not CACHE_ONLY_D:
        ld.close_session()

print(
    f'Done Step 7 (Module D): total_companies={N_total_e}, run_remaining_start={N_e}, candidate_calls={total_cand_calls_d}, '
    f'all_resolved_rows={total_all_resolved_d}, not_all_rows={total_not_all_d}, '
    f'found_CashSTInvst={total_item_found_d.get("CashSTInvst",0)}'
)


rebuild_output_d = FORCE_REFRESH_D or (N_e > 0) or (not STEP7D_ROWS_PATH.exists()) or (not OUTPUT_PATH_COMBINED_D.exists())
if (not rebuild_output_d) and BASE_PATH_D.exists() and OUTPUT_PATH_COMBINED_D.exists():
    rebuild_output_d = BASE_PATH_D.stat().st_mtime > OUTPUT_PATH_COMBINED_D.stat().st_mtime

if (not rebuild_output_d) and OUTPUT_PATH_COMBINED_D.exists():
    try:
        chk = pd.read_parquet(OUTPUT_PATH_COMBINED_D)
        needed_e = {'CashSTInvst'}
        if not needed_e.issubset(set(chk.columns)):
            rebuild_output_d = True
            print('Step 7 rebuild forced: combined output missing Module-C columns.')
    except Exception:
        rebuild_output_d = True
        print('Step 7 rebuild forced: unable to inspect combined output file.')

# -------------------------
# Build final output table
# -------------------------
if rebuild_output_d and STEP7D_ROWS_PATH.exists():
    out_panel_e = pd.read_parquet(STEP7D_ROWS_PATH)
else:
    out_panel_e = pd.DataFrame(new_rows_out_d)

if rebuild_output_d and out_panel_e.empty:
    out_panel_e = pd.DataFrame(columns=['firm_id', 'date'] + TARGET_COLS_D)
elif rebuild_output_d:
    out_panel_e['date'] = pd.to_datetime(out_panel_e['date'], errors='coerce').dt.normalize()
    for c in TARGET_COLS_D:
        if c not in out_panel_e.columns:
            out_panel_e[c] = np.nan
        out_panel_e[c] = pd.to_numeric(out_panel_e[c], errors='coerce')
    out_panel_e = out_panel_e[['firm_id', 'date'] + TARGET_COLS_D]
    out_panel_e = out_panel_e.sort_values(['firm_id', 'date']).drop_duplicates(['firm_id', 'date'], keep='last')

if rebuild_output_d:
    # Start from base euro500 and preserve existing loaded columns from combined output (if present).
    out_d = base_e.copy()
    if OUTPUT_PATH_COMBINED_D.exists():
        old = pd.read_parquet(OUTPUT_PATH_COMBINED_D).copy()
        old['date'] = _d_resolve_asof(old)

        keep_old = [c for c in old.columns if c not in out_d.columns and c not in {'firm_id', 'date', 'firm_id'}]
        if keep_old:
            # Stage 1: preserve on firm_id + date
            if 'firm_id' in out_d.columns and 'firm_id' in old.columns:
                old_firm = old[['firm_id', 'date'] + keep_old].dropna(subset=['firm_id', 'date'])
                old_firm = old_firm.drop_duplicates(['firm_id', 'date'], keep='last')
                out_d = out_d.merge(old_firm, on=['firm_id', 'date'], how='left', suffixes=('', '_oldfirm'))

            # Stage 2 fallback: preserve on firm_id + date
            old_small = old[['firm_id', 'date'] + keep_old].dropna(subset=['firm_id', 'date'])
            old_small = old_small.drop_duplicates(['firm_id', 'date'], keep='last')
            add_old = base_e.merge(old_small, on=['firm_id', 'date'], how='left', suffixes=('', '_oldkey'))

            for c in keep_old:
                old_s = out_d[c] if c in out_d.columns else pd.Series(pd.NA, index=out_d.index, dtype='object')
                old_firm_s = out_d.get(f'{c}_oldfirm', pd.Series(pd.NA, index=out_d.index, dtype='object'))
                old_key_s = add_old.get(f'{c}_oldkey', pd.Series(pd.NA, index=out_d.index, dtype='object'))
                out_d[c] = old_firm_s.fillna(old_key_s).fillna(old_s)
                if f'{c}_oldfirm' in out_d.columns:
                    out_d = out_d.drop(columns=[f'{c}_oldfirm'])

    # Stage 1: preferred merge on firm_id + date
    if 'firm_id' in base_e.columns:
        out_panel_d_firm = out_panel_e.rename(columns={'firm_id': 'firm_id'})
        out_d = out_d.merge(out_panel_d_firm, on=['firm_id', 'date'], how='left', suffixes=('', '_firm'))

    # Stage 2 fallback: merge on firm_id + date
    add_d = base_e.merge(out_panel_e, on=['firm_id', 'date'], how='left', suffixes=('', '_key'))

    for c in TARGET_COLS_D:
        old_s = pd.to_numeric(out_d[c], errors='coerce') if c in out_d.columns else pd.Series(np.nan, index=out_d.index)
        new_firm_s = pd.to_numeric(
            out_d[f'{c}_firm'] if f'{c}_firm' in out_d.columns else pd.Series(np.nan, index=out_d.index),
            errors='coerce'
        )
        new_key_s = pd.to_numeric(
            add_d[f'{c}_key'] if f'{c}_key' in add_d.columns else pd.Series(np.nan, index=out_d.index),
            errors='coerce'
        )
        out_d[c] = new_firm_s.combine_first(new_key_s).combine_first(old_s)
        if f'{c}_firm' in out_d.columns:
            out_d = out_d.drop(columns=[f'{c}_firm'])

    # Keep only canonical beta column in combined output
    if ('beta' not in out_d.columns) and ('beta_3m' in out_d.columns):
        out_d['beta'] = pd.to_numeric(out_d['beta_3m'], errors='coerce')
    if 'beta_3m' in out_d.columns:
        out_d = out_d.drop(columns=['beta_3m'])

    out_d.to_parquet(OUTPUT_PATH_COMBINED_D, index=False)
    euro500_netpayout_df = out_d.copy()

    print('Saved combined output (Step 5 + 6 + 7 + D / Module D):', OUTPUT_PATH_COMBINED_D, 'rows:', len(out_d))
    print('Data Wrangler variable ready: euro500_netpayout_df')
else:
    out_d = pd.read_parquet(OUTPUT_PATH_COMBINED_D)
    euro500_netpayout_df = out_d.copy()
    print('Skipped Step-7D rebuild (already up-to-date):', OUTPUT_PATH_COMBINED_D, 'rows:', len(out_d))
    print('Data Wrangler variable ready: euro500_netpayout_df')

if bad_rows_d:
    bad_df = pd.DataFrame(bad_rows_d)
    if BAD_LOG_PATH_D.exists():
        old = pd.read_csv(BAD_LOG_PATH_D)
        for c in ['firm_id', 'date', 'reason', 'n_candidates']:
            if c not in old.columns:
                old[c] = pd.NA
            if c not in bad_df.columns:
                bad_df[c] = pd.NA
        out_bad = pd.DataFrame.from_records(
            old[['firm_id', 'date', 'reason', 'n_candidates']].to_dict('records')
            + bad_df[['firm_id', 'date', 'reason', 'n_candidates']].to_dict('records'),
            columns=['firm_id', 'date', 'reason', 'n_candidates'],
        )
    else:
        out_bad = bad_df

    out_bad['date'] = pd.to_datetime(out_bad['date'], errors='coerce').dt.normalize()
    out_bad = out_bad.drop_duplicates(subset=['firm_id', 'date', 'reason'], keep='last')
    out_bad.to_csv(BAD_LOG_PATH_D, index=False)
    print('Updated Module-D bad-id log:', BAD_LOG_PATH_D, 'rows:', len(out_bad))







### 5B. Coverage-Analyse Modul D (`CashSTInvst`)

In [ ]:
# Robust path resolution for Step 8B (Module D coverage)
MODULED_OUTPUT_PATH = globals().get('OUTPUT_PATH_COMBINED_D', DATA_DIR / 'euro500_netpayout.parquet')

if not MODULED_OUTPUT_PATH.exists():
    raise FileNotFoundError(f'Missing file: {MODULED_OUTPUT_PATH}')

md = pd.read_parquet(MODULED_OUTPUT_PATH).copy()
if 'date' in md.columns:
    md['date'] = pd.to_datetime(md['date'], errors='coerce')

target_d = 'CashSTInvst'
if target_d not in md.columns:
    raise KeyError('No Module D column found (expected CashSTInvst). Run Step 7D first.')

md[target_d] = pd.to_numeric(md[target_d], errors='coerce')

if 'date' in md.columns and md['date'].notna().any():
    md['year'] = md['date'].dt.year
else:
    md['year'] = pd.NA

overall_d = {
    'rows_total': len(md),
    f'cov_{target_d}': float(md[target_d].notna().mean()) if len(md) else np.nan,
}
display(pd.DataFrame([overall_d]))

if md['year'].notna().any():
    yd = md.groupby('year', as_index=False).agg(
        rows=('year', 'size'),
        cov_CashSTInvst=(target_d, lambda s: s.notna().mean()),
    )

    plt.figure(figsize=(11, 5))
    plt.plot(yd['year'], yd['cov_CashSTInvst'], marker='o', linewidth=1.8, label='CashSTInvst')
    plt.title('Module D yearly coverage (CashSTInvst)')
    plt.xlabel('Year')
    plt.ylabel('Coverage share')
    plt.ylim(0, 1.02)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()

    # Zusatzplot: Module D coverage (Non-Financials only)
    if 'trbc_sector' in md.columns:
        md_nf = md[md['trbc_sector'].astype(str).str.strip().str.lower() != 'financials'].copy()
        if md_nf['year'].notna().any() and not md_nf.empty:
            yd_nf = md_nf.groupby('year', as_index=False).agg(
                rows=('year', 'size'),
                cov_CashSTInvst=(target_d, lambda s: s.notna().mean()),
            )

            plt.figure(figsize=(11, 5))
            plt.plot(yd_nf['year'], yd_nf['cov_CashSTInvst'], marker='o', linewidth=1.8, label='CashSTInvst')
            plt.title('Module D yearly coverage (CashSTInvst, Non-Financials)')
            plt.xlabel('Year')
            plt.ylabel('Coverage share')
            plt.ylim(0, 1.02)
            plt.grid(True, alpha=0.3)
            plt.legend()
            plt.show()
        else:
            print('No valid Non-Financials rows/year information for Module D coverage plot.')
    else:
        print('Column trbc_sector missing; skipping Module D Non-Financials coverage plot.')
else:
    print('No valid year information available for Module D yearly coverage plot.')




## 6. Coverage-Uebersicht ueber alle gezogenen Items (alle Jahre)

In [ ]:
# Step 6 — Coverage summary across NetPayout modules (all years)
STEP6_PATH = DATA_DIR / 'euro500_netpayout.parquet'

if not STEP6_PATH.exists():
    raise FileNotFoundError(f'Missing file: {STEP6_PATH}')

cov = pd.read_parquet(STEP6_PATH).copy()

item_candidates = [
    'BE', 'assets', 'debt',
    'Sales', 'NetIncome', 'GrossProfit',
    'Dividends', 'Buybacks',
    'CashSTInvst',
]
items = [c for c in item_candidates if c in cov.columns]
if not items:
    raise KeyError('No expected NetPayout columns found in euro500_netpayout.parquet.')

for c in items:
    cov[c] = pd.to_numeric(cov[c], errors='coerce')

if 'trbc_sector' in cov.columns:
    mask_nonfin = cov['trbc_sector'].astype(str).str.strip().str.lower() != 'financials'
else:
    mask_nonfin = pd.Series(False, index=cov.index)
    print('Column trbc_sector missing -> non_financials coverage set to NaN.')

item_to_module = {
    'BE': 'Module A',
    'assets': 'Module A',
    'debt': 'Module A',
    'Sales': 'Module B',
    'NetIncome': 'Module B',
    'GrossProfit': 'Module B',
    'Dividends': 'Module C',
    'Buybacks': 'Module C',
    'CashSTInvst': 'Module D',
}

rows = []
for c in items:
    all_cov = float(cov[c].notna().mean()) if len(cov) else np.nan
    if mask_nonfin.any():
        nf_cov = float(cov.loc[mask_nonfin, c].notna().mean())
    else:
        nf_cov = np.nan
    rows.append({
        'module': item_to_module.get(c, 'Other'),
        'item': c,
        'all_equities': all_cov,
        'non_financials': nf_cov,
    })

summary = (
    pd.DataFrame(rows)
    .sort_values(['module', 'item'])
    .reset_index(drop=True)
)

print('Step 6 coverage table (NetPayout modules, all years pooled):')
display(summary)
